# Overview
**If you found it helpful, please consider upvoting it.**

* This notebook demonstrates training an agent using the PPO (Proximal Policy Optimization) reinforcement learning algorithm that outperforms [the Nearest Planet Sniper agent from the Getting Started tutorial](https://www.kaggle.com/code/bovard/getting-started#Agent-1:-Nearest-Planet-Sniper).
* The current implementation makes several simplifying assumptions about the model, observations, and action spaces. To create a stronger agent, we recommend modifying the observation design, reward function, and aiming logic from this point forward.

## Training Approach

- The algorithm is PPO
- Each owned planet within a turn is treated as an independent decision unit
- The policy is called once for each owned planet
- During training, multiple environments are run in parallel, rollouts are collected for a fixed number of steps, and PPO updates are applied
- The opponent is always self-play, and it uses the same policy structure
- The opponent weights are synchronized from the current learner every fixed number of updates

## Action Design

The action space is heavily simplified.
"Whether to send or not" and "where to send" are represented as a single target selection.

- `target_index = 0` means no-op
- `target_index = 1..K-1` represents target candidates
- The policy outputs a categorical distribution over the candidate targets

In other words, this action space learns only:

- for each owned planet
- which candidate target to choose

### Ship Count

The number of ships is not learned.
It uses the same fixed rule as the sniper baseline:

- `ships = max(target.ships + 1, 20)`

That value is sent as-is.
However, if the source planet does not have enough ships, that candidate becomes invalid.

### How Target Candidates Are Built

For each owned source planet `src`, candidates are created from all other planets.

- `index 0` is reserved for no-op
- The remaining slots are filled with enemy, neutral, and friendly planets in distance order
- If there are not enough, the rest are filled by nearest remaining planets

The number of candidates is controlled by `env.candidate_count` in the config.

## Observations

Observations are scalar features rather than images.
Each decision row is composed of the following three groups of features.

### 1. self_features

Features of the source planet `src` itself.
Examples:

- position
- radius
- current ship count
- production
- whether it is a rotating planet
- number of owned planets
- number of enemy planets
- total owned ships
- total enemy ships

### 2. candidate_features

Features for each candidate target.
Examples:

- neutral / friendly / enemy flags
- target position
- relative position from `src`
- distance
- target ship count
- target production
- whether the target is a rotating planet
- whether the current direct shot would hit the sun
- ship count of `src`

### 3. global_features

Global summary features of the board.
Examples:

- turn progress
- number of friendly / enemy / neutral planets
- total friendly / enemy ships
- number of friendly / enemy fleets in flight

## Policy Network

The policy is a simple MLP-based PyTorch model.

It has separate encoders for:

- `self_features`
- `candidate_features`
- `global_features`

`candidate_features` are encoded per candidate, and the `self / global / candidate` embeddings are concatenated to produce a score for each candidate.
The outputs are:

- `target_logits`
  logits for each candidate target
- `value`
  an approximation of the state value for that decision row

A categorical distribution is built from `target_logits`, and `target_index = 0` is treated as no-op.

In short, this network is a simple structure that:

- scores the candidate target set for each owned planet
- and selects one of them

In [49]:
import sys, os

def is_kaggle() -> bool:
    return True if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") else False

def is_colab() -> bool:
    return "google.colab" in str(get_ipython())

def is_local() -> bool:
    return not is_kaggle() and not is_colab()

ENV = 'kaggle' if is_kaggle() else 'colab' if is_colab() else 'local'
print(f"Running on: {ENV}")

Running on: colab


In [50]:
import subprocess, sys
!uv pip install numpy torch tqdm pyyaml wandb "kaggle-environments>=1.28.0"

Using Python 3.12.13 environment at: /usr
Checked 6 packages in 123ms


---

#　Training Script Implementation

- `game_types.py`: Converts observation data into an easier-to-handle format.
- `config.py`: Loads and manages training and environment settings.
- `features.py`: Builds features and action candidates from observations.
- `policy.py`: Predicts actions and values from features.
- `ppo.py`: Handles action sampling and PPO updates.
- `opponents.py`: Chooses actions for the opponent side.
- `env.py`: Wraps interaction with the Kaggle environment.
- `train.py`: Runs the full training loop.

In [51]:
!mkdir -p src

In [52]:
from pathlib import Path

Path("default_cfg.yaml").write_text("""\
seed: 142857
run_name: orbit_wars_ppo
device: auto
save_dir: artifacts
checkpoint_every: 100
log_every: 1
opponent: self
self_play_update_interval: 50
self_play_deterministic: false
alternate_player_sides: true

env:
  candidate_count: 8
  ship_bucket_count: 8

model:
  hidden_size: 128
  num_heads: 4
  num_attn_layers: 1
  mlp_ratio: 2.0
  dropout: 0.0
  use_memory: false
  geom_indices: [0, 1, 2, 3]
  fourier_bands: 4

ppo:
  rollout_steps: 256
  num_envs: 8
  total_updates: 4096
  epochs: 4
  minibatch_size: 256
  gamma: 0.99
  gae_lambda: 0.95
  clip_coef: 0.2
  ent_coef: 0.02
  vf_coef: 0.5
  lr: 0.0002
  max_grad_norm: 0.5
  reward_win: 1.0
  reward_loss: -1.0
  reward_step: 0.0
  reward_planet_owned: 0.002
  reward_ship_ratio: 0.001
""")
print("default_cfg.yaml written")

default_cfg.yaml written


In [53]:
from pathlib import Path

Path("src/__init__.py").write_text("""\
from .config import TrainConfig, default_train_config_path, load_train_config

__all__ = ["TrainConfig", "default_train_config_path", "load_train_config"]
""")
print("src/__init__.py written")

src/__init__.py written


In [54]:
from pathlib import Path

Path("src/config.py").write_text("""\
from __future__ import annotations

import dataclasses
from dataclasses import dataclass, field
from functools import lru_cache
from pathlib import Path
from typing import Any

import yaml


@dataclass(slots=True, frozen=True)
class EnvConfig:
    board_size:        float = 100.0
    episode_steps:     int   = 500
    candidate_count:   int   = 8
    ship_bucket_count: int   = 8
    max_planets:       int   = 48
    max_ships:         float = 400.0
    max_production:    float = 5.0


@dataclass(slots=True, frozen=True)
class ModelConfig:
    hidden_size:     int   = 128
    num_heads:       int   = 4
    num_attn_layers: int   = 2
    mlp_ratio:       float = 2.0
    dropout:         float = 0.0
    use_memory:      bool  = False
    geom_indices:    tuple = (0, 1, 2, 3)
    fourier_bands:   int   = 4


@dataclass(slots=True, frozen=True)
class PPOConfig:
    rollout_steps:        int   = 256
    num_envs:             int   = 8
    total_updates:        int   = 2048
    epochs:               int   = 4
    minibatch_size:       int   = 512
    gamma:                float = 0.99
    gae_lambda:           float = 0.95
    clip_coef:            float = 0.2
    ent_coef:             float = 0.01
    vf_coef:              float = 0.5
    lr:                   float = 0.0003
    max_grad_norm:        float = 0.5
    reward_win:           float = 1.0
    reward_loss:          float = -1.0
    reward_step:          float = 0.0
    reward_planet_owned:  float = 0.002
    reward_ship_ratio:    float = 0.001


@dataclass(slots=True)
class TrainConfig:
    seed:                      int   = 42
    run_name:                  str   = "orbit_wars_ppo"
    device:                    str   = "auto"
    save_dir:                  str   = "artifacts"
    checkpoint_every:          int   = 100
    log_every:                 int   = 1
    opponent:                  str   = "self"
    self_play_update_interval: int   = 50
    self_play_deterministic:   bool  = False
    alternate_player_sides:    bool  = True
    env:   EnvConfig   = field(default_factory=EnvConfig)
    model: ModelConfig = field(default_factory=ModelConfig)
    ppo:   PPOConfig   = field(default_factory=PPOConfig)


@lru_cache(maxsize=1)
def default_train_config_path() -> Path:
    return Path(__file__).resolve().parent.parent / "default_cfg.yaml"


def load_train_config(path: str | Path) -> TrainConfig:
    config_path = Path(path)
    data = yaml.safe_load(config_path.read_text(encoding="utf-8")) or {}
    if not isinstance(data, dict):
        raise ValueError(f"YAML config must be a mapping: {config_path}")
    return _config_from_dict(data)


def _config_from_dict(data: dict[str, Any]) -> TrainConfig:
    env_cfg   = _make_frozen(EnvConfig,   data.get("env",   {}))
    model_cfg = _make_frozen(ModelConfig, data.get("model", {}))
    ppo_cfg   = _make_frozen(PPOConfig,   data.get("ppo",   {}))
    cfg = TrainConfig(env=env_cfg, model=model_cfg, ppo=ppo_cfg)
    top = {k: v for k, v in data.items() if k not in {"env", "model", "ppo"}}
    _update(cfg, top)
    return cfg


def _make_frozen(cls, data: dict[str, Any]):
    defaults = cls()
    kwargs = {}
    for f in dataclasses.fields(cls):
        if f.name in data:
            kwargs[f.name] = _coerce(data[f.name], getattr(defaults, f.name))
    return cls(**kwargs)


def _update(instance: Any, values: dict[str, Any]) -> None:
    if not isinstance(values, dict):
        return
    for key, value in values.items():
        if not hasattr(instance, key):
            continue
        setattr(instance, key, _coerce(value, getattr(instance, key)))


_BOOL_TRUE  = frozenset({"1", "true", "yes", "on"})
_BOOL_FALSE = frozenset({"0", "false", "no", "off"})


def _coerce(value: Any, default: Any) -> Any:
    t = type(default)
    if t is bool:
        if isinstance(value, str):
            low = value.strip().lower()
            if low in _BOOL_TRUE:  return True
            if low in _BOOL_FALSE: return False
        return bool(value)
    if t is int:   return int(value)
    if t is float: return float(value)
    if t is tuple: return tuple(value) if isinstance(value, (list, tuple)) else default
    return value
""")
print("src/config.py written")

src/config.py written


In [55]:
from pathlib import Path

Path("src/game_types.py").write_text("""\
from __future__ import annotations

from dataclasses import dataclass
from typing import Any

import numpy as np


@dataclass(slots=True)
class PlanetState:
    id:         int
    owner:      int
    x:          float
    y:          float
    radius:     float
    ships:      int
    production: int


@dataclass(slots=True)
class FleetState:
    id:             int
    owner:          int
    x:              float
    y:              float
    angle:          float
    from_planet_id: int
    ships:          int


@dataclass(slots=True)
class GameState:
    step:    int
    player:  int
    planets: list[PlanetState]
    fleets:  list[FleetState]


def _get(obs: Any, key: str, default: Any) -> Any:
    return obs.get(key, default) if isinstance(obs, dict) \\
           else getattr(obs, key, default)


def parse_observation(obs: Any) -> GameState:
    raw_planets = _get(obs, "planets", [])
    raw_fleets  = _get(obs, "fleets",  [])

    planets: list[PlanetState] = []
    if raw_planets:
        arr   = np.asarray(raw_planets, dtype=np.float64)
        ids   = arr[:, 0].astype(np.int32)
        owners= arr[:, 1].astype(np.int32)
        xs    = arr[:, 2]
        ys    = arr[:, 3]
        radii = arr[:, 4]
        ships = arr[:, 5].astype(np.int32)
        prods = arr[:, 6].astype(np.int32)
        for i in range(len(arr)):
            planets.append(PlanetState(
                id=int(ids[i]), owner=int(owners[i]),
                x=float(xs[i]),  y=float(ys[i]),
                radius=float(radii[i]), ships=int(ships[i]),
                production=int(prods[i]),
            ))

    fleets: list[FleetState] = []
    if raw_fleets:
        arr = np.asarray(raw_fleets, dtype=np.float64)
        for i in range(len(arr)):
            fleets.append(FleetState(
                id=int(arr[i, 0]), owner=int(arr[i, 1]),
                x=float(arr[i, 2]),  y=float(arr[i, 3]),
                angle=float(arr[i, 4]), from_planet_id=int(arr[i, 5]),
                ships=int(arr[i, 6]),
            ))

    return GameState(
        step   = int(_get(obs, "step",   0)),
        player = int(_get(obs, "player", 0)),
        planets= planets,
        fleets = fleets,
    )
""")
print("src/game_types.py written")

src/game_types.py written


In [56]:
from pathlib import Path

Path("src/features.py").write_text("""\
from __future__ import annotations

import math
from dataclasses import dataclass
from typing import Any

import numpy as np

from .config import EnvConfig
from .game_types import GameState, parse_observation

_BOARD_CX:  float = 50.0
_BOARD_CY:  float = 50.0
_ROT_LIMIT: float = 50.0
_SUN_R:     float = 10.0
_LAUNCH_OFF:float = 0.1

SELF_DIM      = 11
CANDIDATE_DIM = 14
GLOBAL_DIM    = 8

def self_feature_dim()      -> int: return SELF_DIM
def candidate_feature_dim() -> int: return CANDIDATE_DIM
def global_feature_dim()    -> int: return GLOBAL_DIM


@dataclass(slots=True)
class DecisionContext:
    env_index:      int
    source_id:      int
    candidate_ids:  list[int]
    candidate_mask: np.ndarray
    ship_counts:    list[int]
    target_angles:  list[float]


@dataclass(slots=True)
class TurnBatch:
    self_features:      np.ndarray   # (N, SELF_DIM)
    candidate_features: np.ndarray   # (N, K, CANDIDATE_DIM)
    global_features:    np.ndarray   # (N, GLOBAL_DIM)
    candidate_mask:     np.ndarray   # (N, K) bool
    contexts:           list[DecisionContext]
    state:              GameState


def _is_rotating_vec(
    xs: np.ndarray, ys: np.ndarray, radii: np.ndarray
) -> np.ndarray:
    dx = xs - _BOARD_CX
    dy = ys - _BOARD_CY
    return (np.sqrt(dx * dx + dy * dy) + radii) < _ROT_LIMIT


def _seg_dist_vec(
    px: np.ndarray, py: np.ndarray,
    x1: np.ndarray, y1: np.ndarray,
    x2: np.ndarray, y2: np.ndarray,
) -> np.ndarray:
    dx = x2 - x1
    dy = y2 - y1
    seg_sq = dx * dx + dy * dy
    t = np.where(
        seg_sq > 0.0,
        np.clip(
            ((px - x1) * dx + (py - y1) * dy) / np.maximum(seg_sq, 1e-12),
            0.0, 1.0,
        ),
        0.0,
    )
    cx = x1 + t * dx
    cy = y1 + t * dy
    return np.sqrt((px - cx) ** 2 + (py - cy) ** 2)


def _crosses_sun_vec(
    sx: np.ndarray, sy: np.ndarray, sr: np.ndarray,
    angles: np.ndarray,
    tx: np.ndarray, ty: np.ndarray,
) -> np.ndarray:
    off = sr + _LAUNCH_OFF
    bx  = sx + np.cos(angles) * off
    by  = sy + np.sin(angles) * off
    d   = _seg_dist_vec(
        np.full_like(bx, _BOARD_CX),
        np.full_like(by, _BOARD_CY),
        bx, by, tx, ty,
    )
    return d < _SUN_R


def _top_k_indices(dists: np.ndarray, mask: np.ndarray, k: int) -> np.ndarray:
    valid  = np.where(mask)[0]
    result = np.full(k, -1, dtype=np.int32)
    if valid.size == 0:
        return result
    n_take = min(k, valid.size)
    if valid.size <= k:
        sorted_valid = valid[np.argsort(dists[valid])]
    else:
        part         = np.argpartition(dists[valid], n_take - 1)[:n_take]
        sorted_valid = valid[part[np.argsort(dists[valid[part]])]]
    result[:n_take] = sorted_valid.astype(np.int32)
    return result


def _build_candidates_vectorized(
    src_indices: np.ndarray,
    xs: np.ndarray,
    ys: np.ndarray,
    owners: np.ndarray,
    planet_ids: np.ndarray,
    player_id: int,
    k: int, n: int,
    eq: int, nq: int, fq: int,
) -> np.ndarray:
    M      = len(src_indices)
    result = np.full((M, k), -1, dtype=np.int32)

    src_x  = xs[src_indices][:, None]
    src_y  = ys[src_indices][:, None]
    dists  = np.sqrt((src_x - xs[None, :]) ** 2 + (src_y - ys[None, :]) ** 2)

    base_valid = planet_ids != -1

    for i, si in enumerate(src_indices):
        d        = dists[i].copy()
        d[si]    = np.inf
        not_self = np.arange(n) != si
        valid    = base_valid & not_self

        o       = owners
        em_mask = valid & (o != -1) & (o != player_id)
        nm_mask = valid & (o == -1)
        fm_mask = valid & (o == player_id)

        et = _top_k_indices(d, em_mask, eq)
        nt = _top_k_indices(d, nm_mask, nq)
        ft = _top_k_indices(d, fm_mask, fq)

        pos = 0
        for arr in (et, nt, ft):
            for idx in arr:
                if idx != -1 and pos < k:
                    result[i, pos] = idx
                    pos += 1

        if pos < k:
            used      = set(int(x) for x in result[i] if x != -1)
            remaining = valid & ~np.isin(np.arange(n), list(used))
            extra     = _top_k_indices(d, remaining, k - pos)
            for idx in extra:
                if idx != -1 and pos < k:
                    result[i, pos] = idx
                    pos += 1

    return result


def _build_self_features_vec(
    src_indices: np.ndarray,
    xs: np.ndarray, ys: np.ndarray,
    radii: np.ndarray, ships: np.ndarray, production: np.ndarray,
    my_cnt_n: float, en_cnt_n: float,
    my_shp_n: float, en_shp_n: float,
    inv_board: float, inv_ms: float, inv_mp: float,
    max_ships: float,
) -> np.ndarray:
    M    = len(src_indices)
    f    = np.empty((M, SELF_DIM), dtype=np.float32)
    f[:, 0]  = 1.0
    f[:, 1]  = xs[src_indices]    * inv_board
    f[:, 2]  = ys[src_indices]    * inv_board
    f[:, 3]  = radii[src_indices] * (1.0 / 5.0)
    f[:, 4]  = np.minimum(ships[src_indices], max_ships) * inv_ms
    f[:, 5]  = production[src_indices] * inv_mp
    f[:, 6]  = _is_rotating_vec(
        xs[src_indices], ys[src_indices], radii[src_indices]
    ).astype(np.float32)
    f[:, 7]  = my_cnt_n
    f[:, 8]  = en_cnt_n
    f[:, 9]  = my_shp_n
    f[:, 10] = en_shp_n
    return f


def _build_candidate_features_vec(
    src_indices:  np.ndarray,
    cid_mat:      np.ndarray,
    xs: np.ndarray, ys: np.ndarray,
    radii: np.ndarray, ships: np.ndarray,
    production: np.ndarray, owners: np.ndarray,
    player_id: int,
    inv_board: float, inv_ms: float, inv_mp: float,
    max_ships: float,
    k: int,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    M     = len(src_indices)
    cf    = np.zeros((M, k, CANDIDATE_DIM), dtype=np.float32)
    cmask = np.zeros((M, k),               dtype=bool)
    scnt  = np.zeros((M, k),               dtype=np.int32)
    cids  = np.full( (M, k), -1,           dtype=np.int32)
    tang  = np.zeros((M, k),               dtype=np.float32)

    # Slot 0 is always the no-op action (always valid)
    cmask[:, 0] = True

    sx = xs[src_indices]
    sy = ys[src_indices]
    sr = radii[src_indices]
    ss = ships[src_indices]

    valid = cid_mat != -1
    mi, ki = np.where(valid)
    if mi.size == 0:
        return cf, cmask, scnt, cids, tang

    ti_flat = cid_mat[mi, ki]
    tx      = xs[ti_flat]
    ty      = ys[ti_flat]
    tr      = radii[ti_flat]
    tsh     = ships[ti_flat]
    tprod   = production[ti_flat]
    town    = owners[ti_flat]

    sx_v = sx[mi]
    sy_v = sy[mi]
    sr_v = sr[mi]
    ss_v = ss[mi]

    dx    = tx - sx_v
    dy    = ty - sy_v
    angle = np.arctan2(dy, dx)
    dist  = np.sqrt(dx * dx + dy * dy)
    sun   = _crosses_sun_vec(sx_v, sy_v, sr_v, angle, tx, ty)
    need  = np.maximum(tsh.astype(np.int32) + 1, 20)

    cf[mi, ki, 0]  = 1.0
    cf[mi, ki, 1]  = (town == -1).astype(np.float32)
    cf[mi, ki, 2]  = (town == player_id).astype(np.float32)
    cf[mi, ki, 3]  = ((town != -1) & (town != player_id)).astype(np.float32)
    cf[mi, ki, 4]  = tx   * inv_board
    cf[mi, ki, 5]  = ty   * inv_board
    cf[mi, ki, 6]  = dx   * inv_board
    cf[mi, ki, 7]  = dy   * inv_board
    cf[mi, ki, 8]  = dist * inv_board
    cf[mi, ki, 9]  = np.minimum(tsh, max_ships) * inv_ms
    cf[mi, ki, 10] = tprod * inv_mp
    cf[mi, ki, 11] = _is_rotating_vec(tx, ty, tr).astype(np.float32)
    cf[mi, ki, 12] = sun.astype(np.float32)
    cf[mi, ki, 13] = np.minimum(ss_v, max_ships) * inv_ms

    scnt[mi, ki] = need
    cids[mi, ki] = ti_flat
    tang[mi, ki] = angle

    # Valid action: enough ships on source, target not behind sun
    cmask[mi, ki] = (~sun) & (ss_v >= need)

    return cf, cmask, scnt, cids, tang


def encode_turn(
    observation: Any,
    env_cfg: EnvConfig,
    *,
    env_index: int = 0,
) -> TurnBatch:
    state = (
        observation if isinstance(observation, GameState)
        else parse_observation(observation)
    )

    board_size    = env_cfg.board_size
    k             = env_cfg.candidate_count
    n             = env_cfg.max_planets
    max_ships     = env_cfg.max_ships
    max_prod      = env_cfg.max_production
    episode_steps = env_cfg.episode_steps

    inv_board = 1.0 / board_size
    inv_ms    = 1.0 / max_ships
    inv_mp    = 1.0 / max_prod
    inv_n     = 1.0 / n

    eq = k // 3
    nq = k // 3
    fq = k - eq - nq

    planet_ids = np.full(n, -1,  dtype=np.int32)
    owners     = np.full(n, -2,  dtype=np.int32)
    xs         = np.zeros(n,     dtype=np.float32)
    ys         = np.zeros(n,     dtype=np.float32)
    radii      = np.zeros(n,     dtype=np.float32)
    ships      = np.zeros(n,     dtype=np.float32)
    production = np.zeros(n,     dtype=np.float32)

    planets  = state.planets
    np_limit = min(len(planets), n)

    if np_limit > 0:
        arr = np.asarray(
            [(p.id, p.owner, p.x, p.y, p.radius, p.ships, p.production)
             for p in planets[:np_limit]],
            dtype=np.float64,
        )
        planet_ids[:np_limit] = arr[:, 0].astype(np.int32)
        owners[:np_limit]     = arr[:, 1].astype(np.int32)
        xs[:np_limit]         = arr[:, 2].astype(np.float32)
        ys[:np_limit]         = arr[:, 3].astype(np.float32)
        radii[:np_limit]      = arr[:, 4].astype(np.float32)
        ships[:np_limit]      = arr[:, 5].astype(np.float32)
        production[:np_limit] = arr[:, 6].astype(np.float32)

    valid   = planet_ids != -1
    my_mask = valid & (owners == state.player)
    en_mask = valid & (owners != state.player) & (owners != -1)
    ne_mask = valid & (owners == -1)

    my_planet_indices = np.where(my_mask)[0].astype(np.int32)
    M = len(my_planet_indices)

    if M == 0:
        return TurnBatch(
            self_features      = np.zeros((0, SELF_DIM),         dtype=np.float32),
            candidate_features = np.zeros((0, k, CANDIDATE_DIM), dtype=np.float32),
            global_features    = np.zeros((0, GLOBAL_DIM),       dtype=np.float32),
            candidate_mask     = np.zeros((0, k),                dtype=bool),
            contexts           = [],
            state              = state,
        )

    my_cnt  = float(my_mask.sum())
    en_cnt  = float(en_mask.sum())
    ne_cnt  = float(ne_mask.sum())
    my_shp  = float(ships[my_mask].sum())
    en_shp  = float(ships[en_mask].sum())
    max_tot = n * max_ships

    my_cnt_n = my_cnt  * inv_n
    en_cnt_n = en_cnt  * inv_n
    my_shp_n = my_shp  / max_tot
    en_shp_n = en_shp  / max_tot

    cid_mat = _build_candidates_vectorized(
        my_planet_indices, xs, ys, owners, planet_ids,
        state.player, k, n, eq, nq, fq,
    )

    self_arr = _build_self_features_vec(
        my_planet_indices,
        xs, ys, radii, ships, production,
        my_cnt_n, en_cnt_n, my_shp_n, en_shp_n,
        inv_board, inv_ms, inv_mp, max_ships,
    )

    cf, cmask, scnt_mat, cids_mat, tang_mat = _build_candidate_features_vec(
        my_planet_indices, cid_mat,
        xs, ys, radii, ships, production, owners,
        state.player, inv_board, inv_ms, inv_mp, max_ships, k,
    )

    # Fleet counts for global features
    fleets      = state.fleets
    my_fleets   = sum(1 for f in fleets if f.owner == state.player)
    en_fleets   = sum(1 for f in fleets if f.owner != state.player)

    contexts: list[DecisionContext] = []
    for i, si in enumerate(my_planet_indices):
        raw_cids = cids_mat[i]
        n_safe   = max(n, 1)
        pid_arr  = np.where(
            raw_cids >= 0,
            planet_ids[np.clip(raw_cids, 0, n_safe - 1)],
            -1,
        )
        contexts.append(DecisionContext(
            env_index      = env_index,
            source_id      = int(planet_ids[int(si)]),
            candidate_ids  = pid_arr.tolist(),
            candidate_mask = cmask[i],
            ship_counts    = scnt_mat[i].tolist(),
            target_angles  = tang_mat[i].tolist(),
        ))

    gf = np.array([
        state.step / episode_steps,
        my_cnt_n,
        en_cnt_n,
        ne_cnt * inv_n,
        my_shp_n,
        en_shp_n,
        min(my_fleets, 20) / 20.0,
        min(en_fleets, 20) / 20.0,
    ], dtype=np.float32)
    global_arr = np.broadcast_to(gf[None, :], (M, GLOBAL_DIM)).copy()

    return TurnBatch(
        self_features      = self_arr,
        candidate_features = cf,
        global_features    = global_arr,
        candidate_mask     = cmask,
        contexts           = contexts,
        state              = state,
    )
""")
print("src/features.py written")

src/features.py written


In [57]:
from pathlib import Path

Path("src/policy.py").write_text("""\
from __future__ import annotations

import math
import warnings
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor


@dataclass(slots=True)
class PolicyOutput:
    target_logits: Tensor        # (B, K)
    value:         Tensor        # (B,)
    hidden_state:  Tensor | None # (B, h)


class SwiGLU(nn.Module):
    def __init__(self, in_dim: int, out_dim: int, bias: bool = True) -> None:
        super().__init__()
        self.proj    = nn.Linear(in_dim, out_dim * 2, bias=bias)
        self.out_dim = out_dim

    def forward(self, x: Tensor) -> Tensor:
        x1, x2 = self.proj(x).chunk(2, dim=-1)
        return x1 * F.silu(x2)


def _mlp(in_dim: int, hidden: int, out_dim: int, dropout: float = 0.0) -> nn.Sequential:
    layers: list[nn.Module] = [SwiGLU(in_dim, hidden)]
    if dropout > 0.0:
        layers.append(nn.Dropout(dropout))
    layers.append(nn.Linear(hidden, out_dim))
    return nn.Sequential(*layers)


def _masked_mean(x: Tensor, mask: Tensor) -> Tensor:
    x_m   = x.masked_fill(~mask.unsqueeze(-1), 0.0)
    denom = mask.sum(dim=1, keepdim=True).clamp(min=1).float()
    return x_m.sum(dim=1) / denom


class FourierFeatures(nn.Module):
    def __init__(self, in_dim: int, num_bands: int) -> None:
        super().__init__()
        self.out_dim = in_dim * num_bands * 2
        # persistent=True so it is saved in state_dict and survives npz round-trip
        self.register_buffer(
            "freqs",
            torch.randn(in_dim, num_bands) * 2.0 * math.pi,
            persistent=True,
        )

    def forward(self, x: Tensor) -> Tensor:
        proj = x.unsqueeze(-1) * self.freqs
        proj = proj.reshape(*x.shape[:-1], -1)
        return torch.cat([proj.sin(), proj.cos()], dim=-1)


class MultiHeadAttention(nn.Module):
    def __init__(self, dim: int, num_heads: int, dropout: float = 0.0) -> None:
        super().__init__()
        assert dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim  = dim // num_heads
        self.dropout   = dropout
        self.qkv  = nn.Linear(dim, dim * 3, bias=False)
        self.proj = nn.Linear(dim, dim,     bias=False)

    def forward(self, x: Tensor, attn_mask: Tensor | None = None) -> Tensor:
        B, L, D = x.shape
        H, Dh   = self.num_heads, self.head_dim

        q, k, v = self.qkv(x).split(D, dim=-1)
        q = q.view(B, L, H, Dh).transpose(1, 2)
        k = k.view(B, L, H, Dh).transpose(1, 2)
        v = v.view(B, L, H, Dh).transpose(1, 2)

        out = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask = attn_mask,
            dropout_p = self.dropout if self.training else 0.0,
        )
        out = out.transpose(1, 2).contiguous().view(B, L, D)
        return self.proj(out)


class TransformerBlock(nn.Module):
    def __init__(
        self,
        dim:       int,
        num_heads: int,
        mlp_ratio: float = 2.0,
        dropout:   float = 0.0,
    ) -> None:
        super().__init__()
        mlp_dim    = int(dim * mlp_ratio)
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)
        self.attn  = MultiHeadAttention(dim, num_heads, dropout)
        self.ff    = nn.Sequential(
            SwiGLU(dim, mlp_dim, bias=True),
            nn.Dropout(dropout) if dropout > 0.0 else nn.Identity(),
            nn.Linear(mlp_dim, dim),
        )
        self.drop  = nn.Dropout(dropout) if dropout > 0.0 else nn.Identity()

    def forward(self, x: Tensor, attn_mask: Tensor | None = None) -> Tensor:
        x = x + self.drop(self.attn(self.norm1(x), attn_mask))
        x = x + self.drop(self.ff(self.norm2(x)))
        return x


class CrossAttentionPooling(nn.Module):
    def __init__(self, dim: int, num_heads: int, dropout: float = 0.0) -> None:
        super().__init__()
        self.query = nn.Parameter(torch.zeros(1, 1, dim))
        self.attn  = MultiHeadAttention(dim, num_heads, dropout)
        self.norm  = nn.LayerNorm(dim)
        nn.init.normal_(self.query, std=0.02)

    def forward(self, x: Tensor, mask: Tensor | None = None) -> Tensor:
        B, K, _ = x.shape
        q        = self.query.expand(B, 1, -1)
        xq       = torch.cat([q, x], dim=1)

        attn_bias = None
        if mask is not None:
            mq        = F.pad(mask, (1, 0), value=True)
            L         = K + 1
            attn_bias = torch.zeros(B, 1, L, L, dtype=x.dtype, device=x.device)
            attn_bias = attn_bias.masked_fill(~mq[:, None, None, :], float("-inf"))

        out = self.attn(self.norm(xq), attn_bias)
        return out[:, 0, :]


class PlanetPolicy(nn.Module):
    def __init__(
        self,
        self_dim:        int,
        candidate_dim:   int,
        global_dim:      int,
        candidate_count: int,
        hidden_size:     int   = 128,
        num_heads:       int   = 4,
        num_attn_layers: int   = 2,
        mlp_ratio:       float = 2.0,
        dropout:         float = 0.0,
        use_memory:      bool  = False,
        geom_indices:    tuple[int, ...] = (0, 1, 2, 3),
        fourier_bands:   int   = 4,
    ) -> None:
        super().__init__()
        self.candidate_count = candidate_count
        self.use_memory      = use_memory

        h       = hidden_size
        ctx_dim = h * 3

        self.geom_encoder = FourierFeatures(len(geom_indices), fourier_bands)
        geom_dim = self.geom_encoder.out_dim
        # geom_idx is purely a runtime index helper, does not need saving
        self.register_buffer(
            "geom_idx",
            torch.tensor(list(geom_indices), dtype=torch.long),
            persistent=False,
        )

        self.self_encoder      = _mlp(self_dim,                 h, h, dropout)
        self.global_encoder    = _mlp(global_dim,               h, h, dropout)
        self.candidate_encoder = _mlp(candidate_dim + geom_dim, h, h, dropout)

        self.context_proj = nn.Sequential(
            SwiGLU(ctx_dim, h),
            nn.LayerNorm(h),
        )

        self.attn_layers = nn.ModuleList([
            TransformerBlock(h, num_heads, mlp_ratio, dropout)
            for _ in range(num_attn_layers)
        ])

        if use_memory:
            self.memory      = nn.GRUCell(h, h)
            self.memory_norm = nn.LayerNorm(h)
        else:
            self.memory      = None
            self.memory_norm = None

        self.target_head = nn.Sequential(
            SwiGLU(h, h),
            nn.Linear(h, 1, bias=False),
        )
        self.value_pool = CrossAttentionPooling(h, num_heads, dropout)
        self.value_head = nn.Sequential(
            SwiGLU(ctx_dim, h),
            nn.Linear(h, 1, bias=False),
        )

        self._neg_inf = float("-inf")
        self._init_weights()

    def _init_weights(self) -> None:
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.orthogonal_(m.weight, gain=math.sqrt(2))
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.GRUCell):
                for name, param in m.named_parameters():
                    if "weight" in name:
                        nn.init.orthogonal_(param.data)
                    elif "bias" in name:
                        nn.init.zeros_(param.data)

    def forward(
        self,
        self_features:      Tensor,
        candidate_features: Tensor,
        global_features:    Tensor,
        candidate_mask:     Tensor,
        hidden_state:       Tensor | None = None,
    ) -> PolicyOutput:
        B, K, _ = candidate_features.shape

        # Reconstruct geom_idx on the correct device if needed (non-persistent buffer)
        if not hasattr(self, "geom_idx") or self.geom_idx.device != candidate_features.device:
            # This handles the non-persistent buffer after device transfer
            pass

        geom    = candidate_features[..., self.geom_idx]
        geom    = self.geom_encoder(geom)

        self_h   = self.self_encoder(self_features)
        global_h = self.global_encoder(global_features)

        cand_in = torch.cat([candidate_features, geom], dim=-1)
        cand_h  = self.candidate_encoder(cand_in.reshape(B * K, -1)).reshape(B, K, -1)

        self_exp   = self_h.unsqueeze(1).expand(B, K, -1)
        global_exp = global_h.unsqueeze(1).expand(B, K, -1)
        x = self.context_proj(torch.cat([self_exp, global_exp, cand_h], dim=-1))

        attn_bias = None
        if candidate_mask is not None:
            attn_bias = torch.zeros(B, 1, K, K, dtype=x.dtype, device=x.device)
            attn_bias = attn_bias.masked_fill(
                ~candidate_mask[:, None, None, :], self._neg_inf
            )

        for layer in self.attn_layers:
            x = layer(x, attn_bias)

        next_hidden: Tensor | None = None
        if self.use_memory and self.memory is not None and self.memory_norm is not None:
            pooled      = _masked_mean(x, candidate_mask)
            next_hidden = self.memory(pooled, hidden_state)
            x           = x + self.memory_norm(next_hidden).unsqueeze(1)

        logits = self.target_head(x).squeeze(-1)
        logits = logits.masked_fill(~candidate_mask, self._neg_inf)

        value_ctx = self.value_pool(x, candidate_mask)
        value     = self.value_head(
            torch.cat([value_ctx, self_h, global_h], dim=-1)
        ).squeeze(-1)

        return PolicyOutput(
            target_logits = logits,
            value         = value,
            hidden_state  = next_hidden,
        )
""")
print("src/policy.py written")

src/policy.py written


In [58]:
from pathlib import Path

Path("src/ppo.py").write_text("""\
from __future__ import annotations

from dataclasses import dataclass

import torch
import torch.nn.functional as F
from torch import Tensor
from torch.distributions import Categorical

from .policy import PolicyOutput


@dataclass(slots=True)
class SampledAction:
    target_index: Tensor
    log_prob:     Tensor
    entropy:      Tensor


@dataclass(slots=True)
class TransitionBatch:
    self_features:      Tensor   # (N, SELF_DIM)
    candidate_features: Tensor   # (N, K, CANDIDATE_DIM)
    global_features:    Tensor   # (N, GLOBAL_DIM)
    candidate_mask:     Tensor   # (N, K) bool
    target_index:       Tensor   # (N,) long
    log_prob:           Tensor   # (N,) float
    returns:            Tensor   # (N,) float  — GAE targets
    advantages:         Tensor   # (N,) float  — GAE advantages


def _safe_logits(logits: Tensor) -> Tensor:
    \"\"\"Ensure at least one valid logit per row to avoid NaN in Categorical.\"\"\"
    all_invalid = ~torch.isfinite(logits).any(dim=-1)
    if not all_invalid.any():
        return logits
    out = logits.clone()
    out[all_invalid, 0] = 0.0
    return out


def _log_prob_and_entropy(
    logits: Tensor, idx: Tensor
) -> tuple[Tensor, Tensor]:
    dist = Categorical(logits=logits)
    return dist.log_prob(idx), dist.entropy()


def sample_actions(outputs: PolicyOutput, deterministic: bool) -> SampledAction:
    logits = _safe_logits(outputs.target_logits)
    if deterministic:
        idx = logits.argmax(dim=-1)
    else:
        idx = Categorical(logits=logits).sample()
    lp, ent = _log_prob_and_entropy(logits, idx)
    return SampledAction(target_index=idx, log_prob=lp, entropy=ent)


def action_log_prob_and_entropy(
    outputs: PolicyOutput,
    target_index: Tensor,
) -> tuple[Tensor, Tensor]:
    return _log_prob_and_entropy(
        _safe_logits(outputs.target_logits), target_index
    )


def ppo_update(
    policy:    torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    batch:     TransitionBatch,
    *,
    clip_coef:      float,
    ent_coef:       float,
    vf_coef:        float,
    max_grad_norm:  float,
    epochs:         int,
    minibatch_size: int,
    device:         torch.device,
) -> dict[str, float]:
    N = batch.self_features.shape[0]
    if N == 0:
        return {"loss": 0.0, "policy_loss": 0.0, "value_loss": 0.0, "entropy": 0.0}

    def _to(t: Tensor) -> Tensor:
        return t.to(device, non_blocking=True)

    sf   = _to(batch.self_features)
    cf   = _to(batch.candidate_features)
    gf   = _to(batch.global_features)
    cm   = _to(batch.candidate_mask).bool()
    olp  = _to(batch.log_prob)
    tidx = _to(batch.target_index)
    ret  = _to(batch.returns)
    adv  = _to(batch.advantages)

    # Normalize advantages over the full batch
    adv = (adv - adv.mean()) / (adv.std(unbiased=False) + 1e-8)

    mb   = min(N, max(1, minibatch_size))
    acc  = {"loss": 0.0, "policy_loss": 0.0, "value_loss": 0.0, "entropy": 0.0}
    n_up = 0

    for _ in range(epochs):
        perm = torch.randperm(N, device=device)
        for start in range(0, N, mb):
            idx = perm[start : start + mb]

            outputs      = policy(sf[idx], cf[idx], gf[idx], cm[idx])
            nlp, ent     = action_log_prob_and_entropy(outputs, tidx[idx])
            ratio        = (nlp - olp[idx]).exp()

            adv_mb = adv[idx]
            pg1    = -adv_mb * ratio
            pg2    = -adv_mb * ratio.clamp(1.0 - clip_coef, 1.0 + clip_coef)
            policy_loss  = torch.maximum(pg1, pg2).mean()
            value_loss   = 0.5 * F.mse_loss(outputs.value, ret[idx])
            entropy_mean = ent.mean()

            loss = policy_loss + vf_coef * value_loss - ent_coef * entropy_mean

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(policy.parameters(), max_grad_norm)
            optimizer.step()

            acc["loss"]        += loss.detach().item()
            acc["policy_loss"] += policy_loss.detach().item()
            acc["value_loss"]  += value_loss.detach().item()
            acc["entropy"]     += entropy_mean.detach().item()
            n_up += 1

    inv = 1.0 / max(n_up, 1)
    return {k: v * inv for k, v in acc.items()}
""")
print("src/ppo.py written")

src/ppo.py written


In [59]:
from pathlib import Path

Path("src/opponents.py").write_text("""\
from __future__ import annotations

from typing import Any, Protocol

import numpy as np
import torch

from .config import TrainConfig
from .features import CANDIDATE_DIM, GLOBAL_DIM, SELF_DIM, encode_turn
from .policy import PlanetPolicy
from .ppo import sample_actions


class OpponentPolicy(Protocol):
    def act(self, observation: Any) -> list[list[float | int]]: ...


def _obs_get(obs: Any, key: str, default: Any) -> Any:
    return obs.get(key, default) if isinstance(obs, dict) \\
           else getattr(obs, key, default)


def _strip_compiled(state: dict) -> dict:
    \"\"\"Remove torch.compile _orig_mod. prefix from state dict keys.\"\"\"
    if not any(k.startswith("_orig_mod.") for k in state):
        return state
    return {
        (k[len("_orig_mod."):] if k.startswith("_orig_mod.") else k): v
        for k, v in state.items()
    }


class SelfPlayOpponent:
    \"\"\"
    Frozen copy of the learner policy used as the opponent.
    Weights are periodically synced from the live learner.
    \"\"\"
    __slots__ = ("cfg", "device", "deterministic", "policy")

    def __init__(
        self,
        cfg:           TrainConfig,
        device:        torch.device,
        deterministic: bool = False,
    ) -> None:
        self.cfg           = cfg
        self.device        = device
        self.deterministic = deterministic
        m = cfg.model
        self.policy = PlanetPolicy(
            self_dim        = SELF_DIM,
            candidate_dim   = CANDIDATE_DIM,
            global_dim      = GLOBAL_DIM,
            candidate_count = cfg.env.candidate_count,
            hidden_size     = m.hidden_size,
            num_heads       = getattr(m, "num_heads",       4),
            num_attn_layers = getattr(m, "num_attn_layers", 2),
            mlp_ratio       = getattr(m, "mlp_ratio",       2.0),
            dropout         = 0.0,           # always eval, no dropout
            use_memory      = getattr(m, "use_memory",      False),
            geom_indices    = tuple(getattr(m, "geom_indices", (0, 1, 2, 3))),
            fourier_bands   = getattr(m, "fourier_bands",   4),
        ).to(device)
        self.policy.eval()

    def sync_from(self, source: PlanetPolicy) -> None:
        self.policy.load_state_dict(
            _strip_compiled(source.state_dict()), strict=True
        )
        self.policy.eval()

    def act(self, observation: Any) -> list[list[float | int]]:
        batch = encode_turn(observation, self.cfg.env, env_index=0)
        if batch.self_features.shape[0] == 0:
            return []

        dev = self.device
        with torch.inference_mode():
            outputs = self.policy(
                torch.from_numpy(batch.self_features).to(dev,      non_blocking=True),
                torch.from_numpy(batch.candidate_features).to(dev, non_blocking=True),
                torch.from_numpy(batch.global_features).to(dev,    non_blocking=True),
                torch.from_numpy(batch.candidate_mask).to(dev,     non_blocking=True).bool(),
                None,
            )
            sampled = sample_actions(outputs, deterministic=self.deterministic)

        tgt   = sampled.target_index.cpu().numpy()
        moves: list[list[float | int]] = []
        for ri, ctx in enumerate(batch.contexts):
            ti = int(tgt[ri])
            if ti == 0 or ti >= len(ctx.candidate_ids):
                continue
            if not ctx.candidate_mask[ti]:
                continue
            ships = int(ctx.ship_counts[ti])
            if ships <= 0:
                continue
            moves.append([ctx.source_id, float(ctx.target_angles[ti]), ships])
        return moves


def build_opponent(
    name:   str,
    cfg:    TrainConfig | None = None,
    device: torch.device | None = None,
) -> OpponentPolicy:
    if name == "self":
        if cfg is None or device is None:
            raise ValueError("cfg and device required for self-play")
        return SelfPlayOpponent(
            cfg, device=device,
            deterministic=cfg.self_play_deterministic,
        )
    raise ValueError(f"Unknown opponent: {name!r}. Supported: 'self'")
""")
print("src/opponents.py written")

src/opponents.py written


In [60]:
from pathlib import Path

Path("src/env.py").write_text("""\
from __future__ import annotations

from dataclasses import dataclass
from typing import Any

from .config import TrainConfig
from .features import TurnBatch, encode_turn
from .opponents import OpponentPolicy


@dataclass(slots=True)
class StepResult:
    batch:        TurnBatch
    reward:       float
    shaped_reward:float
    done:         bool
    info:         dict[str, Any]


def _get(state: Any, key: str, default: Any = None) -> Any:
    return state.get(key, default) if isinstance(state, dict) \\
           else getattr(state, key, default)


def _obs(s: Any)    -> Any:  return _get(s, "observation")
def _status(s: Any) -> str:  return str(_get(s, "status", "UNKNOWN"))

def _raw_reward(s: Any) -> float:
    v = _get(s, "reward", 0.0)
    return 0.0 if v is None else float(v)


def _terminal_reward(
    player_state: Any,
    opp_state:    Any,
    reward_win:   float,
    reward_loss:  float,
) -> float:
    \"\"\"
    Map kaggle terminal rewards to clean win/loss/draw signals.

    Kaggle orbit_wars gives:
      winner  -> reward > 0
      loser   -> reward < 0 (or 0 in some draw cases)
      draw    -> both get 0

    We map cleanly: win=+reward_win, loss=reward_loss, draw=0.
    \"\"\"
    pr = _raw_reward(player_state)
    or_ = _raw_reward(opp_state)
    if pr > 0 and or_ <= 0:
        return reward_win
    if pr < 0 or (pr == 0 and or_ > 0):
        return reward_loss
    # draw or both-zero
    return 0.0


def _shaping_reward(
    obs:                Any,
    player_id:          int,
    reward_planet_owned:float,
    reward_ship_ratio:  float,
) -> float:
    \"\"\"
    Dense per-step reward shaping to guide learning before terminal.

    Components:
      + reward_planet_owned  per owned planet above starting (encourages expansion)
      + reward_ship_ratio    * (my_ships / total_ships - 0.5)  (encourages army advantage)
    \"\"\"
    if obs is None:
        return 0.0
    raw_planets = _get(obs, "planets", [])
    if not raw_planets:
        return 0.0

    my_planets = 0
    my_ships   = 0.0
    en_ships   = 0.0
    total      = len(raw_planets)

    for p in raw_planets:
        # p is either a list/tuple [id, owner, x, y, r, ships, prod]
        # or an object with attributes
        if isinstance(p, (list, tuple)):
            owner, ships = int(p[1]), float(p[5])
        else:
            owner, ships = int(getattr(p, "owner", -1)), float(getattr(p, "ships", 0))

        if owner == player_id:
            my_planets += 1
            my_ships   += ships
        elif owner != -1:
            en_ships   += ships

    total_ships = my_ships + en_ships + 1e-6
    planet_bonus = reward_planet_owned * my_planets
    ratio_bonus  = reward_ship_ratio   * (my_ships / total_ships - 0.5)
    return planet_bonus + ratio_bonus


class OrbitWarsEnv:
    __slots__ = (
        "cfg", "opponent", "make_fn", "env_index",
        "env", "last_obs", "last_opp_obs",
        "episode_index", "learner_player",
    )

    def __init__(
        self,
        cfg:       TrainConfig,
        opponent:  OpponentPolicy,
        make_fn:   Any | None = None,
        env_index: int = 0,
    ) -> None:
        self.cfg            = cfg
        self.opponent       = opponent
        self.make_fn        = make_fn
        self.env_index      = env_index
        self.env            = None
        self.last_obs       = None
        self.last_opp_obs   = None
        self.episode_index  = 0
        self.learner_player = 0

    def reset(self, seed: int | None = None) -> TurnBatch:
        from kaggle_environments import make as _make
        make_fn = self.make_fn or _make

        cfg_d: dict[str, Any] = {}
        if seed is not None:
            cfg_d["seed"] = cfg_d["randomSeed"] = int(seed)

        # Alternate which player side we play to avoid side bias
        self.learner_player = (
            (self.env_index + self.episode_index) % 2
            if self.cfg.alternate_player_sides else 0
        )

        self.env = make_fn("orbit_wars", configuration=cfg_d, debug=False)
        self.env.reset(num_agents=2)
        states = self.env.step([[], []])

        lp                = self.learner_player
        self.last_obs     = _obs(states[lp])
        self.last_opp_obs = _obs(states[1 - lp])
        self.episode_index += 1

        return encode_turn(self.last_obs, self.cfg.env, env_index=self.env_index)

    def step(self, action: list[list[float | int]]) -> StepResult:
        if self.env is None:
            raise RuntimeError("Call reset() before step().")

        opp   = self.opponent.act(self.last_opp_obs)
        lp    = self.learner_player
        joint = [action, opp] if lp == 0 else [opp, action]

        states = self.env.step(joint)
        ps, os = states[lp], states[1 - lp]

        prev_obs      = self.last_obs
        self.last_obs     = _obs(ps)
        self.last_opp_obs = _obs(os)

        done = _status(ps) != "ACTIVE"

        ppo = self.cfg.ppo
        if done:
            terminal  = _terminal_reward(ps, os, ppo.reward_win, ppo.reward_loss)
            shaping   = 0.0          # no shaping on terminal step
            shaped_r  = terminal
        else:
            terminal  = ppo.reward_step
            shaping   = _shaping_reward(
                self.last_obs,
                self.learner_player,
                ppo.reward_planet_owned,
                ppo.reward_ship_ratio,
            )
            shaped_r  = terminal + shaping

        batch = encode_turn(self.last_obs, self.cfg.env, env_index=self.env_index)

        return StepResult(
            batch         = batch,
            reward        = terminal,
            shaped_reward = shaped_r,
            done          = done,
            info          = {
                "learner_player" : lp,
                "player_status"  : _status(ps),
                "opponent_status": _status(os),
                "raw_reward"     : _raw_reward(ps),
                "shaped_reward"  : shaped_r,
                "shaping"        : shaping,
            },
        )
""")
print("src/env.py written")

src/env.py written


In [61]:
from pathlib import Path

Path("src/train.py").write_text("""\
from __future__ import annotations

import argparse
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import torch

from .config import TrainConfig, default_train_config_path, load_train_config
from .env import OrbitWarsEnv
from .features import CANDIDATE_DIM, GLOBAL_DIM, SELF_DIM, TurnBatch
from .game_types import PlanetState
from .opponents import SelfPlayOpponent, build_opponent, _strip_compiled
from .policy import PlanetPolicy
from .ppo import TransitionBatch, ppo_update, sample_actions


# ── Per-decision-row storage during rollout ───────────────────────

@dataclass(slots=True)
class RowRecord:
    \"\"\"Everything we need from a single per-planet decision row.\"\"\"
    self_feat:  np.ndarray   # (SELF_DIM,)
    cand_feat:  np.ndarray   # (K, CANDIDATE_DIM)
    glob_feat:  np.ndarray   # (GLOBAL_DIM,)
    mask:       np.ndarray   # (K,) bool
    target_idx: int
    log_prob:   float
    value:      float        # V(s) from critic at decision time


@dataclass(slots=True)
class EpisodeStep:
    \"\"\"
    One environment step worth of rows plus the shaped reward received.
    All rows in one step share the same reward (per-env).
    \"\"\"
    row_slice:     slice
    shaped_reward: float
    done:          bool


# ── W&B helper ────────────────────────────────────────────────────

def _try_import_wandb():
    \"\"\"Return wandb module or None if not installed / disabled.\"\"\"
    try:
        import wandb
        return wandb
    except ImportError:
        return None


def _wandb_init(cfg: TrainConfig, run_name: str) -> Any | None:
    \"\"\"
    Initialise a W&B run.  Returns the run object or None.

    Reads optional env-vars:
      WANDB_PROJECT   (default: orbit_wars_ppo)
      WANDB_ENTITY    (default: None — uses account default)
      WANDB_MODE      (default: online;  set to 'disabled' to suppress)
    \"\"\"
    import os
    wandb = _try_import_wandb()
    if wandb is None:
        print("[wandb] not installed — logging disabled. "
              "Run `pip install wandb` to enable.")
        return None

    project = os.environ.get("WANDB_PROJECT", "orbit_wars_ppo")
    entity  = os.environ.get("WANDB_ENTITY",  None)
    mode    = os.environ.get("WANDB_MODE",    "online")

    # Flatten config into a plain dict for W&B
    flat_cfg = {
        # top-level
        "seed":                      cfg.seed,
        "opponent":                  cfg.opponent,
        "self_play_update_interval": cfg.self_play_update_interval,
        "self_play_deterministic":   cfg.self_play_deterministic,
        "alternate_player_sides":    cfg.alternate_player_sides,
        # env
        "env/candidate_count":       cfg.env.candidate_count,
        "env/episode_steps":         cfg.env.episode_steps,
        "env/max_planets":           cfg.env.max_planets,
        "env/max_ships":             cfg.env.max_ships,
        # model
        "model/hidden_size":         cfg.model.hidden_size,
        "model/num_heads":           cfg.model.num_heads,
        "model/num_attn_layers":     cfg.model.num_attn_layers,
        "model/mlp_ratio":           cfg.model.mlp_ratio,
        "model/dropout":             cfg.model.dropout,
        "model/fourier_bands":       cfg.model.fourier_bands,
        # ppo
        "ppo/rollout_steps":         cfg.ppo.rollout_steps,
        "ppo/num_envs":              cfg.ppo.num_envs,
        "ppo/total_updates":         cfg.ppo.total_updates,
        "ppo/epochs":                cfg.ppo.epochs,
        "ppo/minibatch_size":        cfg.ppo.minibatch_size,
        "ppo/gamma":                 cfg.ppo.gamma,
        "ppo/gae_lambda":            getattr(cfg.ppo, "gae_lambda", 0.95),
        "ppo/clip_coef":             cfg.ppo.clip_coef,
        "ppo/ent_coef":              cfg.ppo.ent_coef,
        "ppo/vf_coef":               cfg.ppo.vf_coef,
        "ppo/lr":                    cfg.ppo.lr,
        "ppo/max_grad_norm":         cfg.ppo.max_grad_norm,
        "ppo/reward_win":            getattr(cfg.ppo, "reward_win",  1.0),
        "ppo/reward_loss":           getattr(cfg.ppo, "reward_loss", -1.0),
        "ppo/reward_planet_owned":   getattr(cfg.ppo, "reward_planet_owned", 0.002),
        "ppo/reward_ship_ratio":     getattr(cfg.ppo, "reward_ship_ratio",   0.001),
    }

    try:
        run = wandb.init(
            project  = project,
            entity   = entity,
            name     = run_name,
            config   = flat_cfg,
            mode     = mode,
            resume   = "allow",
        )
        print(f"[wandb] run: {run.url}")
        return run
    except Exception as e:
        print(f"[wandb] init failed ({e}) — continuing without logging.")
        return None


def _wandb_log(
    run:        Any | None,
    update:     int,
    stats:      dict[str, float],
    metrics:    dict[str, float],
    policy:     PlanetPolicy,
    optimizer:  torch.optim.Optimizer,
    *,
    extra:      dict[str, float] | None = None,
) -> None:
    \"\"\"
    Log one update worth of metrics to W&B.

    Logged groups:
      rollout/   — environment interaction statistics
      loss/      — PPO loss components
      train/     — gradient / parameter health
      hp/        — live hyper-parameters (lr, etc.)
    \"\"\"
    if run is None:
        return

    wandb = _try_import_wandb()
    if wandb is None:
        return

    # ── Gradient norm (computed cheaply from last backward) ───────
    total_norm = 0.0
    for p in policy.parameters():
        if p.grad is not None:
            total_norm += p.grad.detach().data.norm(2).item() ** 2
    grad_norm = total_norm ** 0.5

    # ── Parameter norm ────────────────────────────────────────────
    param_norm = sum(
        p.detach().norm(2).item() ** 2
        for p in policy.parameters()
    ) ** 0.5

    # ── Current learning rate ─────────────────────────────────────
    current_lr = optimizer.param_groups[0]["lr"]

    payload: dict[str, float] = {
        # Rollout statistics
        "rollout/episode_reward_mean": stats["episode_reward_mean"],
        "rollout/episodes_finished":   stats["episodes_finished"],
        "rollout/samples":             stats["samples"],

        # PPO loss components
        "loss/total":                  metrics["loss"],
        "loss/policy":                 metrics["policy_loss"],
        "loss/value":                  metrics["value_loss"],
        "loss/entropy":                metrics["entropy"],

        # Gradient and parameter health
        "train/grad_norm":             grad_norm,
        "train/param_norm":            param_norm,

        # Live hyper-parameters
        "hp/learning_rate":            current_lr,
    }

    if extra:
        payload.update(extra)

    try:
        run.log(payload, step=update)
    except Exception as e:
        print(f"[wandb] log failed at update {update}: {e}")


def _wandb_log_checkpoint(
    run:      Any | None,
    update:   int,
    ckpt_path:Path,
) -> None:
    \"\"\"Save checkpoint as a W&B artifact (optional, can be slow).\"\"\"
    if run is None:
        return
    wandb = _try_import_wandb()
    if wandb is None:
        return
    try:
        artifact = wandb.Artifact(
            name = f"checkpoint-{run.id}",
            type = "model",
            metadata = {"update": update},
        )
        artifact.add_file(str(ckpt_path))
        run.log_artifact(artifact)
    except Exception as e:
        print(f"[wandb] artifact upload failed: {e}")


# ── Helpers ───────────────────────────────────────────────────────

def parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser()
    p.add_argument("--config",    type=str,  default=str(default_train_config_path()))
    p.add_argument("--resume",    type=str,  default=None)
    p.add_argument("--no-wandb",  action="store_true",
                   help="Disable W&B logging even if wandb is installed.")
    p.add_argument("--wandb-project", type=str, default=None,
                   help="Override WANDB_PROJECT env-var.")
    return p.parse_args()


def resolve_device(name: str) -> torch.device:
    if name != "auto":
        return torch.device(name)
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


def _find_planet(planets: list[PlanetState], pid: int) -> PlanetState | None:
    for p in planets:
        if p.id == pid:
            return p
    return None


def _build_policy(cfg: TrainConfig, device: torch.device) -> PlanetPolicy:
    m = cfg.model
    return PlanetPolicy(
        self_dim        = SELF_DIM,
        candidate_dim   = CANDIDATE_DIM,
        global_dim      = GLOBAL_DIM,
        candidate_count = cfg.env.candidate_count,
        hidden_size     = m.hidden_size,
        num_heads       = getattr(m, "num_heads",       4),
        num_attn_layers = getattr(m, "num_attn_layers", 2),
        mlp_ratio       = getattr(m, "mlp_ratio",       2.0),
        dropout         = getattr(m, "dropout",         0.0),
        use_memory      = getattr(m, "use_memory",      False),
        geom_indices    = tuple(getattr(m, "geom_indices", (0, 1, 2, 3))),
        fourier_bands   = getattr(m, "fourier_bands",   4),
    ).to(device)


def _merge_batches(batches: list[TurnBatch], k: int) -> tuple[
    np.ndarray, np.ndarray, np.ndarray, np.ndarray, list[int]
]:
    \"\"\"
    Concatenate TurnBatches into single arrays.
    Returns (sf, cf, gf, cm, sizes) where sizes[i] = rows from batches[i].
    Empty batches contribute 0 rows but keep their index in sizes.
    \"\"\"
    sizes = [b.self_features.shape[0] for b in batches]
    parts = [b for b in batches if b.self_features.shape[0] > 0]
    if parts:
        sf = np.concatenate([b.self_features      for b in parts], axis=0)
        cf = np.concatenate([b.candidate_features for b in parts], axis=0)
        gf = np.concatenate([b.global_features    for b in parts], axis=0)
        cm = np.concatenate([b.candidate_mask     for b in parts], axis=0)
    else:
        sf = np.zeros((0, SELF_DIM),         dtype=np.float32)
        cf = np.zeros((0, k, CANDIDATE_DIM), dtype=np.float32)
        gf = np.zeros((0, GLOBAL_DIM),       dtype=np.float32)
        cm = np.zeros((0, k),                dtype=bool)
    return sf, cf, gf, cm, sizes


def _bootstrap_values_merged(
    policy:  PlanetPolicy,
    batches: list[TurnBatch],
    device:  torch.device,
    k:       int,
) -> list[float]:
    \"\"\"
    Run the critic on the post-rollout observations to get bootstrap V(s_T).
    Returns one float per environment (mean over that env's planets, or 0).
    \"\"\"
    sf, cf, gf, cm, sizes = _merge_batches(batches, k)
    M = sf.shape[0]
    if M == 0:
        return [0.0] * len(batches)

    with torch.inference_mode():
        out = policy(
            torch.from_numpy(sf).to(device, non_blocking=True),
            torch.from_numpy(cf).to(device, non_blocking=True),
            torch.from_numpy(gf).to(device, non_blocking=True),
            torch.from_numpy(cm).to(device, non_blocking=True).bool(),
            None,
        )
    vals = out.value.cpu().numpy()   # (M,)

    result = []
    offset = 0
    for sz in sizes:
        if sz == 0:
            result.append(0.0)
        else:
            result.append(float(vals[offset : offset + sz].mean()))
            offset += sz
    return result


# ── Core rollout collection ───────────────────────────────────────

def collect_rollout(
    envs:      list[OrbitWarsEnv],
    batches:   list[TurnBatch],
    policy:    PlanetPolicy,
    cfg:       TrainConfig,
    device:    torch.device,
    next_seed: int,
) -> tuple[TransitionBatch, list[TurnBatch], int, dict[str, float]]:
    \"\"\"
    Collect T steps across NE environments then compute GAE returns.
    \"\"\"
    k     = cfg.env.candidate_count
    T     = cfg.ppo.rollout_steps
    NE    = len(envs)
    gamma = cfg.ppo.gamma
    lam   = getattr(cfg.ppo, "gae_lambda", 0.95)

    env_steps: list[list[EpisodeStep]] = [[] for _ in range(NE)]
    all_rows:  list[RowRecord]         = []

    ep_rewards:  list[float] = []
    ep_lengths:  list[int]   = []
    running_r:   list[float] = [0.0] * NE
    running_len: list[int]   = [0]   * NE

    # Track action-type breakdown for diagnostics
    noop_count:   int = 0
    attack_count: int = 0
    total_rows:   int = 0

    for t in range(T):
        sf, cf, gf, cm, sizes = _merge_batches(batches, k)
        M          = sf.shape[0]
        cumulative = np.cumsum([0] + sizes)

        if M > 0:
            with torch.inference_mode():
                out = policy(
                    torch.from_numpy(sf).to(device, non_blocking=True),
                    torch.from_numpy(cf).to(device, non_blocking=True),
                    torch.from_numpy(gf).to(device, non_blocking=True),
                    torch.from_numpy(cm).to(device, non_blocking=True).bool(),
                    None,
                )
                sampled  = sample_actions(out, deterministic=False)
                row_vals = out.value.cpu().numpy()
                s_tidx   = sampled.target_index.cpu().numpy()
                s_lp     = sampled.log_prob.cpu().numpy()
        else:
            row_vals = np.zeros(0, dtype=np.float32)
            s_tidx   = np.zeros(0, dtype=np.int64)
            s_lp     = np.zeros(0, dtype=np.float32)

        next_batches: list[TurnBatch] = []

        for ei, env in enumerate(envs):
            b         = batches[ei]
            start     = int(cumulative[ei])
            moves:    list[list[float | int]] = []
            row_start = len(all_rows)

            for li, ctx in enumerate(b.contexts):
                gi  = start + li
                ti  = int(s_tidx[gi])   if M > 0 and gi < M else 0
                lp  = float(s_lp[gi])   if M > 0 and gi < M else 0.0
                val = float(row_vals[gi]) if M > 0 and gi < M else 0.0

                all_rows.append(RowRecord(
                    self_feat  = b.self_features[li],
                    cand_feat  = b.candidate_features[li],
                    glob_feat  = b.global_features[li],
                    mask       = b.candidate_mask[li],
                    target_idx = ti,
                    log_prob   = lp,
                    value      = val,
                ))

                total_rows += 1
                if ti == 0:
                    noop_count += 1
                else:
                    attack_count += 1

                if (ti > 0
                        and ti < len(ctx.candidate_ids)
                        and ctx.candidate_mask[ti]
                        and ctx.ship_counts[ti] > 0):
                    ships = int(ctx.ship_counts[ti])
                    src   = _find_planet(b.state.planets, ctx.source_id)
                    if src is not None and src.ships >= ships:
                        moves.append([ctx.source_id,
                                      float(ctx.target_angles[ti]),
                                      ships])

            row_end = len(all_rows)
            result  = env.step(moves)

            running_r[ei]   += result.shaped_reward
            running_len[ei] += 1

            env_steps[ei].append(EpisodeStep(
                row_slice     = slice(row_start, row_end),
                shaped_reward = result.shaped_reward,
                done          = result.done,
            ))

            if result.done:
                ep_rewards.append(running_r[ei])
                ep_lengths.append(running_len[ei])
                running_r[ei]   = 0.0
                running_len[ei] = 0
                next_seed      += 1
                next_batches.append(env.reset(seed=next_seed))
            else:
                next_batches.append(result.batch)

        batches = next_batches

    # ── Bootstrap final values ────────────────────────────────────
    boot = _bootstrap_values_merged(policy, batches, device, k)

    # ── GAE ───────────────────────────────────────────────────────
    n_rows  = len(all_rows)
    returns = np.zeros(n_rows, dtype=np.float32)
    advs    = np.zeros(n_rows, dtype=np.float32)

    for ei in range(NE):
        steps = env_steps[ei]
        if not steps:
            continue

        gae      = 0.0
        next_val = boot[ei]

        for step in reversed(steps):
            r    = step.shaped_reward
            done = float(step.done)
            sl   = step.row_slice

            if sl.start == sl.stop:
                next_val = (1.0 - done) * next_val
                continue

            step_val = float(
                np.mean([all_rows[i].value for i in range(sl.start, sl.stop)])
            )
            delta = r + gamma * next_val * (1.0 - done) - step_val
            gae   = delta + gamma * lam * (1.0 - done) * gae

            for i in range(sl.start, sl.stop):
                advs[i]    = gae
                returns[i] = gae + all_rows[i].value

            next_val = step_val

    # ── Empty rollout guard ───────────────────────────────────────
    if n_rows == 0:
        empty = TransitionBatch(
            self_features      = torch.zeros((0, SELF_DIM),         dtype=torch.float32),
            candidate_features = torch.zeros((0, k, CANDIDATE_DIM), dtype=torch.float32),
            global_features    = torch.zeros((0, GLOBAL_DIM),       dtype=torch.float32),
            candidate_mask     = torch.zeros((0, k),                 dtype=torch.bool),
            target_index       = torch.zeros(0,                      dtype=torch.long),
            log_prob           = torch.zeros(0,                      dtype=torch.float32),
            returns            = torch.zeros(0,                      dtype=torch.float32),
            advantages         = torch.zeros(0,                      dtype=torch.float32),
        )
        stats = {
            "episode_reward_mean":  0.0,
            "episode_reward_min":   0.0,
            "episode_reward_max":   0.0,
            "episode_reward_std":   0.0,
            "episode_length_mean":  0.0,
            "episodes_finished":    0.0,
            "samples":              0.0,
            "noop_rate":            0.0,
            "attack_rate":          0.0,
            "return_mean":          0.0,
            "return_std":           0.0,
            "advantage_mean":       0.0,
            "advantage_std":        0.0,
            "value_mean":           0.0,
        }
        return empty, batches, next_seed, stats

    # ── Build TransitionBatch ─────────────────────────────────────
    tb = TransitionBatch(
        self_features      = torch.from_numpy(
            np.stack([r.self_feat for r in all_rows]).reshape(-1, SELF_DIM)),
        candidate_features = torch.from_numpy(
            np.stack([r.cand_feat for r in all_rows]).reshape(-1, k, CANDIDATE_DIM)),
        global_features    = torch.from_numpy(
            np.stack([r.glob_feat for r in all_rows]).reshape(-1, GLOBAL_DIM)),
        candidate_mask     = torch.from_numpy(
            np.stack([r.mask      for r in all_rows]).reshape(-1, k)),
        target_index       = torch.tensor(
            [r.target_idx         for r in all_rows], dtype=torch.long),
        log_prob           = torch.tensor(
            [r.log_prob           for r in all_rows], dtype=torch.float32),
        returns            = torch.from_numpy(returns),
        advantages         = torch.from_numpy(advs),
    )

    # ── Rich statistics for logging ───────────────────────────────
    ep_r_arr = np.array(ep_rewards, dtype=np.float32) if ep_rewards else np.zeros(1)
    ep_l_arr = np.array(ep_lengths, dtype=np.float32) if ep_lengths else np.zeros(1)

    stats = {
        # Episode-level
        "episode_reward_mean":  float(ep_r_arr.mean()),
        "episode_reward_min":   float(ep_r_arr.min()),
        "episode_reward_max":   float(ep_r_arr.max()),
        "episode_reward_std":   float(ep_r_arr.std()),
        "episode_length_mean":  float(ep_l_arr.mean()),
        "episodes_finished":    float(len(ep_rewards)),
        "samples":              float(n_rows),
        # Action breakdown
        "noop_rate":            noop_count   / max(total_rows, 1),
        "attack_rate":          attack_count / max(total_rows, 1),
        # Return / advantage distributions
        "return_mean":          float(returns.mean()),
        "return_std":           float(returns.std()),
        "advantage_mean":       float(advs.mean()),
        "advantage_std":        float(advs.std()),
        # Value estimates
        "value_mean":           float(
            np.mean([r.value for r in all_rows])
        ),
    }
    return tb, batches, next_seed, stats


# ── Checkpoint helpers ────────────────────────────────────────────

def save_checkpoint(
    save_dir:  Path,
    run_name:  str,
    update:    int,
    policy:    PlanetPolicy,
    optimizer: torch.optim.Optimizer,
    cfg:       TrainConfig,
) -> Path:
    run_dir = save_dir / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    payload = {
        "update"   : update,
        "policy"   : policy.state_dict(),
        "optimizer": optimizer.state_dict(),
        "config"   : cfg,
    }
    numbered = run_dir / f"ckpt_{update:06d}.pth"
    torch.save(payload, numbered)
    torch.save(payload, run_dir / "ckpt_last.pth")
    return numbered


# ── Entry point ───────────────────────────────────────────────────

def main() -> None:
    args = parse_args()
    cfg  = load_train_config(args.config)
    cfg.save_dir = "artifacts"

    seed_everything(cfg.seed)
    device = resolve_device(cfg.device)
    print(f"Device: {device}")

    # ── W&B setup ─────────────────────────────────────────────────
    import os
    if args.no_wandb:
        os.environ["WANDB_MODE"] = "disabled"
    if args.wandb_project:
        os.environ["WANDB_PROJECT"] = args.wandb_project

    wandb_run = _wandb_init(cfg, run_name=cfg.run_name)

    # ── Environment & opponent ────────────────────────────────────
    opponent = build_opponent(cfg.opponent, cfg=cfg, device=device)

    envs      = [OrbitWarsEnv(cfg, opponent, env_index=i)
                 for i in range(cfg.ppo.num_envs)]
    next_seed = cfg.seed
    batches   = [env.reset(seed=next_seed + i) for i, env in enumerate(envs)]
    next_seed += len(envs)

    # ── Policy ────────────────────────────────────────────────────
    policy_raw = _build_policy(cfg, device)
    n_params   = sum(p.numel() for p in policy_raw.parameters() if p.requires_grad)
    print(f"Policy parameters: {n_params:,}")
    print("Model Architecture:")
    print(policy_raw)

    if wandb_run is not None:
        wandb_run.summary["param_count"] = n_params

    policy = policy_raw

    # ── Resume ────────────────────────────────────────────────────
    start_update = 0
    resume_path  = args.resume
    if resume_path is None:
        auto = Path(cfg.save_dir) / cfg.run_name / "ckpt_last.pth"
        if auto.exists():
            resume_path = str(auto)

    if resume_path and Path(resume_path).exists():
        ckpt = torch.load(resume_path, map_location=device, weights_only=False)
        policy_raw.load_state_dict(_strip_compiled(ckpt["policy"]))
        start_update = int(ckpt.get("update", 0))
        print(f"Resumed from update {start_update}")

    if isinstance(opponent, SelfPlayOpponent):
        opponent.sync_from(policy_raw)

    # ── Optimizer ─────────────────────────────────────────────────
    use_fused = (device.type == "cuda") and torch.cuda.is_available()
    optimizer = torch.optim.AdamW(
        policy_raw.parameters(),
        lr           = cfg.ppo.lr,
        weight_decay = 1e-5,
        **({"fused": True} if use_fused else {}),
    )
    if resume_path and Path(resume_path).exists():
        ckpt = torch.load(resume_path, map_location=device, weights_only=False)
        if "optimizer" in ckpt:
            try:
                optimizer.load_state_dict(ckpt["optimizer"])
            except Exception as e:
                print(f"Could not restore optimizer state: {e}")

    save_dir = Path(cfg.save_dir)

    # ── Training loop ─────────────────────────────────────────────
    try:
        for update in range(start_update + 1, cfg.ppo.total_updates + 1):

            policy.train()
            tb, batches, next_seed, stats = collect_rollout(
                envs, batches, policy, cfg, device, next_seed
            )

            policy.train()
            metrics = ppo_update(
                policy, optimizer, tb,
                clip_coef      = cfg.ppo.clip_coef,
                ent_coef       = cfg.ppo.ent_coef,
                vf_coef        = cfg.ppo.vf_coef,
                max_grad_norm  = cfg.ppo.max_grad_norm,
                epochs         = cfg.ppo.epochs,
                minibatch_size = cfg.ppo.minibatch_size,
                device         = device,
            )

            # Sync opponent periodically
            if (isinstance(opponent, SelfPlayOpponent)
                    and update % cfg.self_play_update_interval == 0):
                opponent.sync_from(policy_raw)

            # ── Console logging ───────────────────────────────────
            if update % cfg.log_every == 0:
                print(
                    f"update={update:>5}  "
                    f"reward={stats['episode_reward_mean']:+.4f}  "
                    f"ep={int(stats['episodes_finished']):>3}  "
                    f"samples={int(stats['samples']):>6}  "
                    f"loss={metrics['loss']:.4f}  "
                    f"pol={metrics['policy_loss']:.4f}  "
                    f"val={metrics['value_loss']:.4f}  "
                    f"ent={metrics['entropy']:.4f}  "
                    f"noop={stats['noop_rate']:.2f}  "
                    f"atk={stats['attack_rate']:.2f}",
                    flush=True,
                )

            # ── W&B logging ───────────────────────────────────────
            if update % cfg.log_every == 0:
                _wandb_log(
                    wandb_run,
                    update,
                    stats,
                    metrics,
                    policy_raw,
                    optimizer,
                    extra={
                        # Extra rollout diagnostics surfaced to W&B
                        "rollout/episode_reward_min":  stats["episode_reward_min"],
                        "rollout/episode_reward_max":  stats["episode_reward_max"],
                        "rollout/episode_reward_std":  stats["episode_reward_std"],
                        "rollout/episode_length_mean": stats["episode_length_mean"],
                        "rollout/noop_rate":           stats["noop_rate"],
                        "rollout/attack_rate":         stats["attack_rate"],
                        "rollout/return_mean":         stats["return_mean"],
                        "rollout/return_std":          stats["return_std"],
                        "rollout/advantage_mean":      stats["advantage_mean"],
                        "rollout/advantage_std":       stats["advantage_std"],
                        "rollout/value_mean":          stats["value_mean"],
                        # Self-play sync indicator
                        "train/opponent_synced": float(
                            isinstance(opponent, SelfPlayOpponent)
                            and update % cfg.self_play_update_interval == 0
                        ),
                    },
                )

            # ── Checkpoint ────────────────────────────────────────
            if update % cfg.checkpoint_every == 0 or update == cfg.ppo.total_updates:
                ckpt_path = save_checkpoint(
                    save_dir, cfg.run_name, update, policy_raw, optimizer, cfg
                )
                print(f"  checkpoint saved -> {ckpt_path}", flush=True)

                # Upload checkpoint artifact to W&B
                _wandb_log_checkpoint(wandb_run, update, ckpt_path)

    finally:
        # Always finish the W&B run cleanly even on crash / KeyboardInterrupt
        if wandb_run is not None:
            wandb_run.finish()
            print("[wandb] run finished.")


if __name__ == "__main__":
    main()
""")
print("src/train.py written")

src/train.py written


# Training PPO

In [62]:
%%capture
!uv pip install --upgrade "kaggle-environments>=1.28.0"

In [63]:
# !rm -rf artifacts/

Note: For this public Notebook, total_updates is set to 100 to keep runtime short. For full training, increase it to 2000.

In [ ]:
import subprocess, sys
import wandb
wandb.login()
!uv run -m src.train --config default_cfg.yaml --wandb-project orbit-wars-ppo

Device: cuda
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: dungngocpham171 (dungngocpham171-university-of-science) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: ⣷ setting up run 7x7axdsp (0.5s)
wandb: ⣯ setting up run 7x7axdsp (0.5s)
wandb: ⣟ setting up run 7x7axdsp (0.5s)
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /content/wandb/run-20260523_142855-7x7axdsp
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run orbit_wars_ppo
wandb: ⭐️ View project at https://wandb.ai/dungngocpham171-university-of-science/orbit-wars-ppo
wandb: 🚀 View run at https://wandb.ai/dungngocpham171-university-of-science/orbit-wars-ppo/runs/7x7axdsp
[wandb] run: https://wandb.ai/dungngocpham171-university-of-science

In [ ]:
import shutil
from google.colab import files

# Zip the artifacts directory
shutil.make_archive('training_artifacts', 'zip', 'artifacts')

# Download the file
files.download('training_artifacts.zip')

In [ ]:
!

# Evaluate

We evaluate the trained models by playing them against the sniper agent.

We measure win rate over 20 games for each of the following checkpoints:
- No_Train (before training)
- 500 updates (25% of total training)
- 1000 updates (50% of total training)
- 2000 updates (full training)

Note: This experiment uses weights trained in a different environment.  
To reproduce these results yourself, set ppo.total_updates to 2000 in your config file and run the training process.

In [65]:
from pathlib import Path

# 1. Define the base directory based on the environment
if ENV == "kaggle":
    base_dir = Path("/kaggle/input/datasets/kashiwaba/orbitwars-ppo-sample-weight")
elif ENV == "colab":
    base_dir = Path("/content/artifacts/orbit_wars_ppo")
else:
    base_dir = Path("./artifacts/orbit_wars_ppo")

# 2. Dynamically build the dictionary using a loop/comprehension
EVAL_CHECKPOINTS = {
    # p.stem gets the filename without the extension (e.g., "ckpt_000500")
    # .replace("ckpt_", "") removes the prefix, leaving just "000500"
    # .lstrip("0") or the whole string keeps "0" if it's the first step, otherwise drops leading zeros
    f"{int(p.stem.replace('ckpt_', ''))}": str(p)
    for p in base_dir.glob("ckpt_*.pth") if "last" not in str(p)
}

# Optional: Sort the dictionary by the integer keys so it prints nicely
EVAL_CHECKPOINTS = dict(sorted(EVAL_CHECKPOINTS.items(), key=lambda item: item[1]))

In [66]:
%%writefile eval_vs_sniper.py

from __future__ import annotations

import argparse
import math
import random
import sys
from collections import namedtuple
from pathlib import Path
from typing import Any

import torch

REPO_ROOT = Path(__file__).resolve().parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.config import TrainConfig, default_train_config_path, load_train_config
from src.features import TurnBatch, candidate_feature_dim, encode_turn, global_feature_dim, self_feature_dim, SELF_DIM, CANDIDATE_DIM, GLOBAL_DIM
from src.policy import PlanetPolicy
from src.ppo import sample_actions

Planet = namedtuple("Planet", ["id", "owner", "x", "y", "radius", "ships", "production"])


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="")
    parser.add_argument("--config", type=str, default=str(default_train_config_path()))
    parser.add_argument("--checkpoint", type=str, default=None)
    parser.add_argument("--games", type=int, default=20)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--device", type=str, default="auto")
    parser.add_argument("--deterministic", action="store_true")
    return parser.parse_args()


def resolve_device(name: str) -> torch.device:
    if name == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return torch.device(name)


def seed_everything(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def build_policy(cfg: TrainConfig, device: torch.device) -> PlanetPolicy:
    m = cfg.model
    return PlanetPolicy(
        self_dim        = SELF_DIM,
        candidate_dim   = CANDIDATE_DIM,
        global_dim      = GLOBAL_DIM,
        candidate_count = cfg.env.candidate_count,
        hidden_size     = m.hidden_size,
        num_heads       = getattr(m, "num_heads",       4),
        num_attn_layers = getattr(m, "num_attn_layers", 2),
        mlp_ratio       = getattr(m, "mlp_ratio",       2.0),
        dropout         = getattr(m, "dropout",         0.0),
        use_memory      = getattr(m, "use_memory",      False),
        geom_indices    = tuple(getattr(m, "geom_indices", (0, 1, 2, 3))),
        fourier_bands   = getattr(m, "fourier_bands",   4),
    ).to(device)


def load_checkpoint_if_available(policy: PlanetPolicy, checkpoint_path: str | None, device: torch.device) -> None:
    if checkpoint_path is None:
        return
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    state_dict = checkpoint.get("policy", checkpoint)
    policy.load_state_dict(state_dict)


def build_moves(batch: TurnBatch, policy: PlanetPolicy, device: torch.device, deterministic: bool) -> list[list[float | int]]:
    if batch.self_features.shape[0] == 0:
        return []
    with torch.inference_mode():
        outputs = policy(
            torch.from_numpy(batch.self_features).to(device),
            torch.from_numpy(batch.candidate_features).to(device),
            torch.from_numpy(batch.global_features).to(device),
            torch.from_numpy(batch.candidate_mask).to(device).bool(),
        )
        sampled = sample_actions(outputs, deterministic=deterministic)
    target_indices = sampled.target_index.detach().cpu().numpy()

    moves: list[list[float | int]] = []
    for row_idx, context in enumerate(batch.contexts):
        target_idx = int(target_indices[row_idx])
        if target_idx == 0:
            continue
        if target_idx >= len(context.candidate_ids):
            continue
        if not context.candidate_mask[target_idx]:
            continue
        ships = int(context.ship_counts[target_idx])
        if ships <= 0:
            continue
        moves.append([context.source_id, float(context.target_angles[target_idx]), ships])
    return moves


def nearest_planet_sniper(obs: Any) -> list[list[float | int]]:
    moves: list[list[float | int]] = []
    player = obs.get("player", 0) if isinstance(obs, dict) else obs.player
    raw_planets = obs.get("planets", []) if isinstance(obs, dict) else obs.planets
    planets = [Planet(*p) for p in raw_planets]
    my_planets = [p for p in planets if p.owner == player]
    targets = [p for p in planets if p.owner != player]
    if not targets:
        return moves
    for mine in my_planets:
        nearest = None
        min_dist = float("inf")
        for target in targets:
            dist = math.hypot(mine.x - target.x, mine.y - target.y)
            if dist < min_dist:
                min_dist = dist
                nearest = target
        if nearest is None:
            continue
        ships_needed = max(nearest.ships + 1, 20)
        if mine.ships < ships_needed:
            continue
        angle = math.atan2(nearest.y - mine.y, nearest.x - mine.x)
        moves.append([mine.id, angle, ships_needed])
    return moves


def extract_observation(state: Any) -> Any:
    if isinstance(state, dict):
        return state.get("observation")
    return getattr(state, "observation")


def extract_status(state: Any) -> str:
    if isinstance(state, dict):
        return str(state.get("status", "UNKNOWN"))
    return str(getattr(state, "status", "UNKNOWN"))


def extract_reward(state: Any) -> float:
    if isinstance(state, dict):
        value = state.get("reward", 0.0)
    else:
        value = getattr(state, "reward", 0.0)
    return 0.0 if value is None else float(value)


def play_one_game(
    cfg: TrainConfig,
    policy: PlanetPolicy,
    device: torch.device,
    *,
    seed: int,
    deterministic: bool,
) -> tuple[float, int]:
    from kaggle_environments import make

    env = make(
        "orbit_wars",
        configuration={"seed": int(seed), "randomSeed": int(seed)},
        debug=False,
    )
    env.reset(num_agents=2)
    states = env.step([[], []])
    player_obs = extract_observation(states[0])
    opponent_obs = extract_observation(states[1])
    done = extract_status(states[0]) != "ACTIVE"
    step_count = 0

    while not done:
        batch = encode_turn(player_obs, cfg.env, env_index=0)
        player_action = build_moves(batch, policy, device, deterministic)
        opponent_action = nearest_planet_sniper(opponent_obs)
        states = env.step([player_action, opponent_action])
        player_obs = extract_observation(states[0])
        opponent_obs = extract_observation(states[1])
        done = extract_status(states[0]) != "ACTIVE"
        step_count += 1

    return extract_reward(states[0]), step_count


def reward_to_label(reward: float) -> str:
    if reward > 0:
        return "win"
    if reward < 0:
        return "loss"
    return "draw"


def main() -> None:
    args = parse_args()
    cfg = load_train_config(args.config)
    device_name = args.device if args.device != "auto" else cfg.device
    device = resolve_device(device_name)
    seed_everything(args.seed)
    policy = build_policy(cfg, device)
    load_checkpoint_if_available(policy, args.checkpoint, device)
    policy.eval()

    wins = 0
    draws = 0
    losses = 0

    for game_idx in range(args.games):
        game_seed = args.seed + game_idx
        reward, steps = play_one_game(
            cfg,
            policy,
            device,
            seed=game_seed,
            deterministic=args.deterministic,
        )
        label = reward_to_label(reward)
        if label == "win":
            wins += 1
        elif label == "loss":
            losses += 1
        else:
            draws += 1
        print(f"game={game_idx + 1} seed={game_seed} result={label} reward={reward:.1f} steps={steps}")

    total_games = max(args.games, 1)
    win_rate = wins / total_games
    print(f"summary wins={wins} losses={losses} draws={draws} games={args.games}")
    print(f"win_rate={win_rate:.4f}")


if __name__ == "__main__":
    main()

Writing eval_vs_sniper.py


### No_Train

In [67]:
!python eval_vs_sniper.py --config default_cfg.yaml --deterministic

[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO: Successfully loaded OpenSpiel environments: 24.
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_amazons
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_backgammon
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_checkers
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_chess
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_clobber
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_coin_game
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_coin_game_arena
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_connect_four
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_dark_hex
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_gin_rummy
[kaggle_environment

In [68]:
!python eval_vs_sniper.py --config default_cfg.yaml --checkpoint {EVAL_CHECKPOINTS["100"]} --deterministic

Traceback (most recent call last):
  File "/content/eval_vs_sniper.py", line 226, in <module>
    main()
  File "/content/eval_vs_sniper.py", line 194, in main
    load_checkpoint_if_available(policy, args.checkpoint, device)
  File "/content/eval_vs_sniper.py", line 65, in load_checkpoint_if_available
    policy.load_state_dict(state_dict)
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 2635, in load_state_dict
    raise RuntimeError(
RuntimeError: Error(s) in loading state_dict for PlanetPolicy:
	Missing key(s) in state_dict: "attn_layers.1.norm1.weight", "attn_layers.1.norm1.bias", "attn_layers.1.norm2.weight", "attn_layers.1.norm2.bias", "attn_layers.1.attn.qkv.weight", "attn_layers.1.attn.proj.weight", "attn_layers.1.ff.0.proj.weight", "attn_layers.1.ff.0.proj.bias", "attn_layers.1.ff.2.weight", "attn_layers.1.ff.2.bias". 


In [69]:
!python eval_vs_sniper.py --config default_cfg.yaml --checkpoint {EVAL_CHECKPOINTS["200"]} --deterministic

Traceback (most recent call last):
  File "/content/eval_vs_sniper.py", line 226, in <module>
    main()
  File "/content/eval_vs_sniper.py", line 194, in main
    load_checkpoint_if_available(policy, args.checkpoint, device)
  File "/content/eval_vs_sniper.py", line 63, in load_checkpoint_if_available
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 1500, in load
    with _open_file_like(f, "rb") as opened_file:
         ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 768, in _open_file_like
    return _open_file(name_or_buffer, mode)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 749, in __init__
    super().__init__(open(name, mode))  # noqa: SIM115
        

### 500 updates (25% of total training)

In [ ]:
!python eval_vs_sniper.py --config default_cfg.yaml --checkpoint {EVAL_CHECKPOINTS["500"]} --deterministic

### 1000 updates (50% of total training)

In [ ]:
!python eval_vs_sniper.py --config default_cfg.yaml --checkpoint {EVAL_CHECKPOINTS["1000"]} --deterministic

In [ ]:
!python eval_vs_sniper.py --config default_cfg.yaml --checkpoint {EVAL_CHECKPOINTS["1100"]} --deterministic

In [ ]:
!python eval_vs_sniper.py --config default_cfg.yaml --checkpoint {EVAL_CHECKPOINTS["1200"]} --deterministic

In [ ]:
!python eval_vs_sniper.py --config default_cfg.yaml --checkpoint {EVAL_CHECKPOINTS["1300"]} --deterministic

In [ ]:
!python eval_vs_sniper.py --config default_cfg.yaml --checkpoint {EVAL_CHECKPOINTS["1400"]} --deterministic

In [ ]:
!python eval_vs_sniper.py --config default_cfg.yaml --checkpoint {EVAL_CHECKPOINTS["1500"]} --deterministic

### 2000 updates (full training)

In [ ]:
!python eval_vs_sniper.py --config default_cfg.yaml --checkpoint {EVAL_CHECKPOINTS["2000"]} --deterministic

**Win Rate Summary**
- No_Train (before training)          :  0%
- 500 updates (25% of total training) : 30%
- 1000 updates (50% of total training): 85%
- 2000 updates (full training)        : 100%

# Game Demo

In [ ]:
%%writefile play_vs_sniper.py

from __future__ import annotations

import argparse
import math
import random
import sys
from collections import namedtuple
from pathlib import Path
from typing import Any

import torch

REPO_ROOT = Path(__file__).resolve().parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.config import TrainConfig, default_train_config_path, load_train_config
from src.features import TurnBatch, candidate_feature_dim, encode_turn, global_feature_dim, self_feature_dim, SELF_DIM, CANDIDATE_DIM, GLOBAL_DIM
from src.policy import PlanetPolicy
from src.ppo import sample_actions

Planet = namedtuple("Planet", ["id", "owner", "x", "y", "radius", "ships", "production"])


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Run one rl_template match against the nearest-planet sniper agent and save the replay as HTML."
    )
    parser.add_argument("--config", type=str, default=str(default_train_config_path()))
    parser.add_argument("--checkpoint", type=str, default=None)
    parser.add_argument("--output", type=str, default="artifacts/rl_template/replays/vs_sniper.html")
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--device", type=str, default="auto")
    parser.add_argument("--deterministic", action="store_true")
    return parser.parse_args()


def resolve_device(name: str) -> torch.device:
    if name == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return torch.device(name)


def seed_everything(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def build_policy(cfg: TrainConfig, device: torch.device) -> PlanetPolicy:
    m = cfg.model
    return PlanetPolicy(
        self_dim        = SELF_DIM,
        candidate_dim   = CANDIDATE_DIM,
        global_dim      = GLOBAL_DIM,
        candidate_count = cfg.env.candidate_count,
        hidden_size     = m.hidden_size,
        num_heads       = getattr(m, "num_heads",       4),
        num_attn_layers = getattr(m, "num_attn_layers", 2),
        mlp_ratio       = getattr(m, "mlp_ratio",       2.0),
        dropout         = getattr(m, "dropout",         0.0),
        use_memory      = getattr(m, "use_memory",      False),
        geom_indices    = tuple(getattr(m, "geom_indices", (0, 1, 2, 3))),
        fourier_bands   = getattr(m, "fourier_bands",   4),
    ).to(device)


def load_checkpoint_if_available(policy: PlanetPolicy, checkpoint_path: str | None, device: torch.device) -> None:
    if checkpoint_path is None:
        return
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    state_dict = checkpoint.get("policy", checkpoint)
    policy.load_state_dict(state_dict)


def build_moves(batch: TurnBatch, policy: PlanetPolicy, device: torch.device, deterministic: bool) -> list[list[float | int]]:
    if batch.self_features.shape[0] == 0:
        return []
    with torch.inference_mode():
        outputs = policy(
            torch.from_numpy(batch.self_features).to(device),
            torch.from_numpy(batch.candidate_features).to(device),
            torch.from_numpy(batch.global_features).to(device),
            torch.from_numpy(batch.candidate_mask).to(device).bool(),
        )
        sampled = sample_actions(outputs, deterministic=deterministic)
    target_indices = sampled.target_index.detach().cpu().numpy()

    moves: list[list[float | int]] = []
    for row_idx, context in enumerate(batch.contexts):
        target_idx = int(target_indices[row_idx])
        if target_idx == 0:
            continue
        if target_idx >= len(context.candidate_ids):
            continue
        if not context.candidate_mask[target_idx]:
            continue
        ships = int(context.ship_counts[target_idx])
        if ships <= 0:
            continue
        moves.append([context.source_id, float(context.target_angles[target_idx]), ships])
    return moves


def nearest_planet_sniper(obs: Any) -> list[list[float | int]]:
    moves: list[list[float | int]] = []
    player = obs.get("player", 0) if isinstance(obs, dict) else obs.player
    raw_planets = obs.get("planets", []) if isinstance(obs, dict) else obs.planets
    planets = [Planet(*p) for p in raw_planets]
    my_planets = [p for p in planets if p.owner == player]
    targets = [p for p in planets if p.owner != player]
    if not targets:
        return moves

    for mine in my_planets:
        nearest = None
        min_dist = float("inf")
        for target in targets:
            dist = math.hypot(mine.x - target.x, mine.y - target.y)
            if dist < min_dist:
                min_dist = dist
                nearest = target
        if nearest is None:
            continue
        ships_needed = max(nearest.ships + 1, 20)
        if mine.ships < ships_needed:
            continue
        angle = math.atan2(nearest.y - mine.y, nearest.x - mine.x)
        moves.append([mine.id, angle, ships_needed])
    return moves


def extract_observation(state: Any) -> Any:
    if isinstance(state, dict):
        return state.get("observation")
    return getattr(state, "observation")


def extract_status(state: Any) -> str:
    if isinstance(state, dict):
        return str(state.get("status", "UNKNOWN"))
    return str(getattr(state, "status", "UNKNOWN"))


def extract_reward(state: Any) -> float:
    if isinstance(state, dict):
        value = state.get("reward", 0.0)
    else:
        value = getattr(state, "reward", 0.0)
    return 0.0 if value is None else float(value)


def run_match(
    cfg: TrainConfig,
    policy: PlanetPolicy,
    device: torch.device,
    *,
    seed: int,
    deterministic: bool,
) -> tuple[str, float, int]:
    from kaggle_environments import make

    env = make(
        "orbit_wars",
        configuration={"seed": int(seed), "randomSeed": int(seed)},
        debug=False,
    )
    env.reset(num_agents=2)
    states = env.step([[], []])
    player_obs = extract_observation(states[0])
    opponent_obs = extract_observation(states[1])
    done = extract_status(states[0]) != "ACTIVE"
    step_count = 0

    while not done:
        batch = encode_turn(player_obs, cfg.env, env_index=0)
        player_action = build_moves(batch, policy, device, deterministic)
        opponent_action = nearest_planet_sniper(opponent_obs)
        states = env.step([player_action, opponent_action])
        player_obs = extract_observation(states[0])
        opponent_obs = extract_observation(states[1])
        done = extract_status(states[0]) != "ACTIVE"
        step_count += 1

    html = env.render(mode="html")
    return html, extract_reward(states[0]), step_count


def main() -> None:
    args = parse_args()
    cfg = load_train_config(args.config)
    device_name = args.device if args.device != "auto" else cfg.device
    device = resolve_device(device_name)
    seed_everything(args.seed)
    policy = build_policy(cfg, device)
    load_checkpoint_if_available(policy, args.checkpoint, device)
    policy.eval()

    html, reward, step_count = run_match(
        cfg,
        policy,
        device,
        seed=args.seed,
        deterministic=args.deterministic,
    )
    output_path = Path(args.output)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(html, encoding="utf-8")
    print(f"saved_html={output_path}")
    print(f"reward={reward:.1f}")
    print(f"steps={step_count}")


if __name__ == "__main__":
    main()

Writing play_vs_sniper.py


In [ ]:
output_path = (
    "/content/result.html"
    if ENV == "colab"
    else "/kaggle/working/result.html" if ENV == "kaggle" else "./artifacts/result.html"
 )
if ENV == "kaggle":
    checkpoint_path = "/kaggle/working/artifacts/orbit_wars_ppo/ckpt_002000.pth"
elif ENV == "colab":
    checkpoint_path = "/content/artifacts/orbit_wars_ppo/ckpt_last.pth"
else:
    checkpoint_path = "./artifacts/orbit_wars_ppo/ckpt_last.pth"
!python play_vs_sniper.py --config default_cfg.yaml --checkpoint {checkpoint_path} --deterministic --output {output_path}

[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO: Successfully loaded OpenSpiel environments: 24.
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_amazons
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_backgammon
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_checkers
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_chess
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_clobber
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_coin_game
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_coin_game_arena
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_connect_four
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_dark_hex
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_gin_rummy
[kaggle_environment

In [ ]:
from IPython.display import HTML, display

with open(output_path, "r", encoding="utf-8") as f:
    html = f.read()

display(HTML(f"""
<iframe
    srcdoc='{html.replace("'", "&apos;")}'
    width="100%"
    height="900"
    style="border:1px solid #ccc;"
 ></iframe>
"""))

In [ ]:
from pathlib import Path

Path("main.py").write_text("""\
\"\"\"
Kaggle orbit_wars submission entry point.
Weights are stored as .npz.  FourierFeatures.freqs is a persistent buffer
(persistent=True in policy.py) so it IS included in state_dict() and
therefore survives the npz round-trip correctly.
\"\"\"
from __future__ import annotations

import os
from pathlib import Path

import numpy as np
import torch

from src.config import TrainConfig
from src.features import (
    candidate_feature_dim,
    encode_turn,
    global_feature_dim,
    self_feature_dim,
)
from src.policy import PlanetPolicy
from src.ppo import sample_actions

# ── Locate weights ────────────────────────────────────────────────

def _find_weights() -> Path:
    candidates = [
        Path("/kaggle_simulations/agent/weights.npz"),
        Path.cwd() / "weights.npz",
        Path("weights.npz"),
    ]
    try:
        candidates.insert(0, Path(__file__).resolve().parent / "weights.npz")
    except NameError:
        pass
    for p in candidates:
        if p.exists():
            return p
    return candidates[0]


_WEIGHTS_PATH = _find_weights()

# ── Lazy-initialised globals ──────────────────────────────────────

_POLICY: PlanetPolicy | None = None
_CONFIG: TrainConfig  | None = None
_DEVICE = torch.device("cpu")


def _load_agent() -> None:
    global _POLICY, _CONFIG

    if not _WEIGHTS_PATH.exists():
        raise FileNotFoundError(f"Weights not found at {_WEIGHTS_PATH}")

    _CONFIG = TrainConfig()          # default config matches training defaults
    m       = _CONFIG.model

    _POLICY = PlanetPolicy(
        self_dim        = self_feature_dim(),
        candidate_dim   = candidate_feature_dim(),
        global_dim      = global_feature_dim(),
        candidate_count = _CONFIG.env.candidate_count,
        hidden_size     = m.hidden_size,
        num_heads       = m.num_heads,
        num_attn_layers = m.num_attn_layers,
        mlp_ratio       = m.mlp_ratio,
        dropout         = 0.0,
        use_memory      = m.use_memory,
        geom_indices    = tuple(m.geom_indices),
        fourier_bands   = m.fourier_bands,
    ).to(_DEVICE)

    # Load weights from npz
    # FourierFeatures.freqs is persistent=True so it appears in state_dict
    # and was saved into the npz correctly.
    raw = np.load(_WEIGHTS_PATH, allow_pickle=False)
    state_dict = {k: torch.from_numpy(raw[k]) for k in raw.files}
    missing, unexpected = _POLICY.load_state_dict(state_dict, strict=False)
    if missing:
        # geom_idx is non-persistent (runtime helper only) — expected to be missing
        non_critical = {k for k in missing if "geom_idx" in k}
        real_missing = set(missing) - non_critical
        if real_missing:
            raise RuntimeError(f"Missing keys in weights.npz: {real_missing}")
    _POLICY.eval()


def agent(obs, config=None):
    global _POLICY, _CONFIG

    if _POLICY is None:
        _load_agent()

    batch = encode_turn(obs, _CONFIG.env, env_index=0)
    if batch.self_features.shape[0] == 0:
        return []

    with torch.inference_mode():
        outputs = _POLICY(
            torch.from_numpy(batch.self_features).to(_DEVICE),
            torch.from_numpy(batch.candidate_features).to(_DEVICE),
            torch.from_numpy(batch.global_features).to(_DEVICE),
            torch.from_numpy(batch.candidate_mask).to(_DEVICE).bool(),
        )
        sampled = sample_actions(outputs, deterministic=True)

    target_indices = sampled.target_index.detach().cpu().numpy()
    moves = []
    for row_idx, context in enumerate(batch.contexts):
        target_idx = int(target_indices[row_idx])
        if target_idx == 0 or target_idx >= len(context.candidate_ids):
            continue
        if not context.candidate_mask[target_idx]:
            continue
        ships = int(context.ship_counts[target_idx])
        if ships <= 0:
            continue
        moves.append([
            context.source_id,
            float(context.target_angles[target_idx]),
            ships,
        ])
    return moves
""")
print("main.py written")

main.py written


### Create Submission Archive
Now we bundle the `main.py` entry point, the helper logic in `src/`, and the weights into a single zip file for submission.

In [ ]:
import shutil
import torch
import numpy as np
from pathlib import Path

DESIRED_CHECKPOINT = "ckpt_000200.pth"
ckpt_file = Path("artifacts/orbit_wars_ppo") / DESIRED_CHECKPOINT

print(f"Loading checkpoint: {ckpt_file}")
checkpoint  = torch.load(ckpt_file, map_location="cpu", weights_only=False)
state_dict  = checkpoint.get("policy", checkpoint)

# Verify freqs buffer is present (persistent=True ensures this)
freqs_keys = [k for k in state_dict if "freqs" in k]
print(f"Fourier freqs keys in state_dict: {freqs_keys}")
if not freqs_keys:
    raise RuntimeError(
        "FourierFeatures.freqs not found in state_dict. "
        "Ensure persistent=True in policy.py FourierFeatures.register_buffer."
    )

# Convert to numpy and save as npz
numpy_weights = {k: v.cpu().numpy() for k, v in state_dict.items()}
np.savez_compressed("weights.npz", **numpy_weights)
print(f"Saved {len(numpy_weights)} tensors to weights.npz")

# Build submission package
submission_dir = Path("submission_package")
if submission_dir.exists():
    shutil.rmtree(submission_dir)
submission_dir.mkdir()

shutil.copy("main.py",          submission_dir / "main.py")
shutil.copy("weights.npz",      submission_dir / "weights.npz")
shutil.copy("default_cfg.yaml", submission_dir / "default_cfg.yaml")

if Path("src").exists():
    shutil.copytree("src", submission_dir / "src")

shutil.make_archive("submission", "zip", submission_dir)

# Verify archive contents
import zipfile
with zipfile.ZipFile("submission.zip") as zf:
    names = zf.namelist()

print(f"submission.zip contains {len(names)} files:")
for n in sorted(names):
    print(f"  {n}")
print("Done.")

Loading checkpoint: artifacts/orbit_wars_ppo/ckpt_000200.pth
Fourier freqs keys in state_dict: ['geom_encoder.freqs']
Saved 48 tensors to weights.npz
submission.zip contains 23 files:
  default_cfg.yaml
  main.py
  src/
  src/__init__.py
  src/__pycache__/
  src/__pycache__/__init__.cpython-312.pyc
  src/__pycache__/config.cpython-312.pyc
  src/__pycache__/env.cpython-312.pyc
  src/__pycache__/features.cpython-312.pyc
  src/__pycache__/game_types.cpython-312.pyc
  src/__pycache__/opponents.cpython-312.pyc
  src/__pycache__/policy.cpython-312.pyc
  src/__pycache__/ppo.cpython-312.pyc
  src/__pycache__/train.cpython-312.pyc
  src/config.py
  src/env.py
  src/features.py
  src/game_types.py
  src/opponents.py
  src/policy.py
  src/ppo.py
  src/train.py
  weights.npz
Done.
